In [ ]:
# 1. Install standard modern Selenium (4.x includes built-in driver/browser management)
!pip install selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.8/511.8 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 10.4 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import Select
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
import pandas as pd
import time
import random
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from google.colab import drive
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
with open("/content/drive/MyDrive/my.flutter.2025/Instant_ai_training/SPRINT_01/DATA.csv","r") as f:
  df=pd.read_csv(f)
cities = df['City.Name'].unique().tolist()

In [ ]:
df["room_type"].value_counts()

,count
room_type,
Entire home/apt,32648
Private room,18693
Shared room,366


In [ ]:
df['bedrooms'].value_counts().sort_values(ascending=False)

,count
bedrooms,
1,36333
2,9290
0,4485
3,1477
4,96
5,10
9,10
6,2
10,2


In [ ]:
# filter number of bedrooms by room_type = Entire home/apt and count them
df[df['room_type'] == 'Entire home/apt']['bedrooms'].value_counts()

,count
bedrooms,
1,18489
2,8650
0,4026
3,1410
4,67
5,4
6,2


In [ ]:
df_copy = df[(df['room_type'] == 'Entire home/apt') & (df['bedrooms'] == 2)].copy()
df_copy

,ID,City.Name,Day.type,realSum,room_type,room_shared,room_private,person_capacity,host_is_superhost,multi,...,guest_satisfaction_overall,bedrooms,dist,metro_dist,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat
36356,591,athens,weekdays,81.081714,Entire home/apt,False,False,2,False,0,...,40,2,0.645994,0.614437,373.834226,14.095104,421.777164,31.669596,23.73117,37.97080
36357,1297,athens,weekdays,185.128770,Entire home/apt,False,False,4,False,0,...,40,2,2.546158,0.655264,67.022833,2.527039,99.349435,7.459760,23.73358,37.99845
36358,1409,athens,weekdays,92.798725,Entire home/apt,False,False,4,False,1,...,20,2,2.937913,0.758379,57.872178,2.182022,85.494093,6.419417,23.73372,38.00199
36359,1438,athens,weekdays,92.798725,Entire home/apt,False,False,3,False,0,...,80,2,1.161515,1.064091,126.861854,4.783219,182.200284,13.680706,23.74338,37.96723
36360,1681,athens,weekdays,125.137675,Entire home/apt,False,False,5,False,0,...,40,2,2.272546,0.106206,68.690918,2.589933,105.230591,7.901353,23.75636,37.98754
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45462,517,paris,weekends,195.498183,Entire home/apt,False,False,2,False,0,...,100,2,3.432411,0.116077,446.342721,21.703447,867.041037,39.998625,2.31479,48.87463
45463,2637,paris,weekends,516.823562,Entire home/apt,False,False,2,False,1,...,100,2,5.704736,0.178499,224.442989,10.913557,439.936551,20.295299,2.28073,48.83722
45464,3045,paris,weekends,207.381862,Entire home/apt,False,False,2,False,0,...,100,2,3.644568,0.193911,299.068461,14.542226,641.114138,29.576090,2.35224,48.88935
45465,1696,vienna,weekdays,278.147862,Entire home/apt,False,False,2,False,0,...,100,2,1.826678,0.444272,116.232687,8.324088,170.394517,4.073404,16.39706,48.20349


In [ ]:
cities

['Amsterdam',
 'Athens',
 'Barcelona',
 'Berlin',
 'Budapest',
 'Lisbon',
 'London',
 'Paris',
 'Rome',
 'Vienna']

In [ ]:
# 1. Install standard modern Selenium (4.x includes built-in driver/browser management)
!pip install selenium

import time
import random
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

# --- ROBUST RATING/FAV EXTRACTOR ---
def get_details_and_fav(driver, wait):
    rating, reviews, is_fav = "N/A", "0", 0

    try:
        # Locate the review anchor (link or button)
        try:
            review_link = wait.until(EC.presence_of_element_located(
                (By.XPATH, "//a[contains(@href, 'reviews')] | //button[contains(., 'reviews')]")
            ))
        except:
            # Fallback for "New" listings
            try:
                h1_parent = driver.find_element(By.TAG_NAME, "h1").find_element(By.XPATH, "./..").text
                if "New" in h1_parent: return "New", "0", 0
            except: pass
            return rating, reviews, is_fav

        # Get text content (cleanly)
        link_text = review_link.get_attribute("textContent").strip()
        clean_link = " ".join(link_text.split())

        # Get parent text (for standard listings where rating is outside link)
        parent_text = review_link.find_element(By.XPATH, "./..").get_attribute("textContent").strip()
        clean_parent = " ".join(parent_text.split())

        # --- SCENARIO A: GUEST FAVORITE ---
        if "Guest favorite" in clean_link or "Guest favorite" in clean_parent:
            is_fav = 1
            # Rating logic for fav
            r_match = re.search(r"(\d\.\d{1,2})", clean_parent)
            if r_match: rating = r_match.group(1)
            # Review logic for fav
            rev_match = re.search(r"(\d+)\s*Review", clean_parent, re.IGNORECASE)
            if rev_match: reviews = rev_match.group(1)

        # --- SCENARIO B: STANDARD ---
        else:
            is_fav = 0
            if "New" in clean_parent:
                rating = "New"
            else:
                r_match = re.search(r"(\d\.\d{1,2})", clean_parent)
                if r_match: rating = r_match.group(1)

            # Try link first, then parent for reviews
            rev_match = re.search(r"(\d+)\s*review", clean_link, re.IGNORECASE)
            if not rev_match:
                rev_match = re.search(r"(\d+)\s*review", clean_parent, re.IGNORECASE)
            if rev_match: reviews = rev_match.group(1)

    except: pass
    return rating, reviews, is_fav

all_data1 = []
city_count = 0

for city in cities:
    city_count += 1

    # --- MAIN SETUP ---
    URL = f"https://www.airbnb.com/s/{city}/homes?refinement_paths%5B%5D=%2Fhomes&date_picker_type=calendar&search_type=filter_change&flexible_trip_lengths%5B%5D=one_week&monthly_start_date=2026-02-01&monthly_length=3&monthly_end_date=2026-05-01&price_filter_input_type=2&channel=EXPLORE&price_filter_num_nights=5&min_bedrooms=2&selected_filter_order%5B%5D=min_bedrooms%3A2&selected_filter_order%5B%5D=room_types%3AEntire%20home%2Fapt&update_selected_filters=true&query={city}&search_mode=regular_search&room_types%5B%5D=Entire%20home%2Fapt"

    chrome_options = Options()
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--start-maximized")

    # In Selenium 4, leaving webdriver.Chrome() without a path automatically invokes
    # Selenium Manager to download and use the correct standalone Chrome binary.
    driver = webdriver.Chrome(options=chrome_options)

    target_count = 50  # Target number of items to scrape per city
    # Initialize a local counter
    items_scraped_this_city = 0
    try:
        driver.get(URL)
        wait = WebDriverWait(driver, 8)

        while items_scraped_this_city < target_count:
            print(f"\n=== SCRAPING CITY {city_count}/{len(cities)}: {city} ===\n")
            print(f"--- Scraping Page (Collected: {len(all_data1)}) ---")

            try:
                wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div[data-testid='card-container']")))
            except: break

            listings = driver.find_elements(By.CSS_SELECTOR, "div[data-testid='card-container']")
            listing_urls = []
            for l in listings:
                try:
                    url = l.find_element(By.TAG_NAME, "a").get_attribute("href")
                    listing_urls.append(url)
                except: continue

            main_window = driver.current_window_handle

            for url in listing_urls:
                if items_scraped_this_city >= target_count: break

                try:
                    driver.execute_script(f"window.open('{url}', '_blank');")
                    driver.switch_to.window(driver.window_handles[-1])
                    wait.until(EC.presence_of_element_located((By.TAG_NAME, "h1")))

                    # A. Name
                    try: name = driver.find_element(By.TAG_NAME, "h1").text
                    except: name = "No Title"

                    # B. Superhost
                    is_superhost = "No"
                    try:
                        driver.execute_script("window.scrollBy(0, 500);")
                        time.sleep(0.5)
                        body_text = driver.find_element(By.TAG_NAME, "body").text
                        if "Superhost" in body_text: is_superhost = "Yes"
                    except: pass
                    # C. Price (Robust Fallback Strategy)
                    price = "N/A"
                    price_selectors = [
                        # 1. Target by the container test ID inside the sidebar (Most stable)
                        "//div[@data-testid='book-it-default-sidebar']//span[contains(text(), '$')]",
                        # 2. Target specific text content containing the price per night structure
                        "//span[contains(text(), 'per night')]/preceding-sibling::span",
                        # 3. Fallback to any element containing the primary price display structure
                        "//div[@data-plugin-in-point-id='BOOK_IT_SIDEBAR']//span[contains(text(), '$')]"
                    ]

                    for selector in price_selectors:
                        try:
                            # Wait up to 3 seconds for the specific price element to appear
                            el = WebDriverWait(driver, 3).until(
                                EC.presence_of_element_located((By.XPATH, selector))
                            )
                            price_text = el.text.strip()
                            if price_text:
                                price = price_text
                                break # Exit the loop if we found a valid price string
                        except:
                            continue

                    # 4. Ultimate safety net: If still N/A, extract it from the page title
                    # (Airbnb listing titles usually display as "$123 night · Listing Name...")
                    if price == "N/A":
                        try:
                            title_text = driver.title
                            price_match = re.search(r"(\$\d[\d,.]*)", title_text)
                            if price_match:
                                price = price_match.group(1)
                        except:
                            pass


                    # D. Rating/Fav (Robust)
                    rating, reviews, is_fav = get_details_and_fav(driver, wait)

                    all_data1.append({
                        "Name": name, "city": city, "Price": price, "Superhost": is_superhost,
                        "is_fav": is_fav, "Rating": rating, "Reviews": reviews, "URL": url
                    })

                    print(f"[{len(all_data1)}] {price} |{city}| ★{rating} ({reviews}) | Fav: {is_fav} | SH: {is_superhost}|")
                    time.sleep(random.uniform(1.0, 2.0))

                    driver.close()
                    driver.switch_to.window(main_window)

                    # Increment the counter for successfully scraped items
                    items_scraped_this_city += 1

                except Exception:
                    if len(driver.window_handles) > 1:
                        driver.close()
                        driver.switch_to.window(main_window)
                    continue

            if items_scraped_this_city < target_count:
                try:
                    next_btn = driver.find_element(By.CSS_SELECTOR, "a[aria-label='Next']")
                    driver.execute_script("arguments[0].click();", next_btn)
                    time.sleep(4)
                except: break

    finally:
        print(f"\nCompleted! Total scraped: {len(all_data1)}")
        driver.quit()


=== SCRAPING CITY 1/10: Amsterdam ===

--- Scraping Page (Collected: 0) ---
[1] $1,202 |Amsterdam| ★4.97 (97322) | Fav: 1 | SH: Yes|

=== SCRAPING CITY 1/10: Amsterdam ===

--- Scraping Page (Collected: 1) ---
[2] $1,350 |Amsterdam| ★4.74 (347) | Fav: 0 | SH: No|
[3] $2,065 |Amsterdam| ★4.86 (86242) | Fav: 1 | SH: Yes|
[4] $3,031 |Amsterdam| ★4.95 (95194) | Fav: 1 | SH: Yes|
[5] $1,877 |Amsterdam| ★4.67 (201) | Fav: 0 | SH: No|
[6] $2,207 |Amsterdam| ★4.91 (91626) | Fav: 1 | SH: Yes|
[7] $2,746 |Amsterdam| ★4.99 (99194) | Fav: 1 | SH: Yes|
[8] $2,230 |Amsterdam| ★4.99 (99242) | Fav: 1 | SH: Yes|
[9] $2,169 |Amsterdam| ★5.0 (024) | Fav: 1 | SH: Yes|
[10] $1,983 |Amsterdam| ★4.79 (7994) | Fav: 1 | SH: Yes|
[11] $1,706 |Amsterdam| ★4.86 (70) | Fav: 0 | SH: Yes|
[12] $1,079 |Amsterdam| ★4.98 (98403) | Fav: 1 | SH: Yes|
[13] $2,347 |Amsterdam| ★N/A (0) | Fav: 0 | SH: No|
[14] $1,600 |Amsterdam| ★4.56 (233) | Fav: 0 | SH: No|
[15] $2,105 |Amsterdam| ★4.95 (95327) | Fav: 1 | SH: Yes|
[16] $2

In [ ]:
df_scraped1 = pd.DataFrame(all_data1)
df_scraped1

,Name,city,Price,Superhost,is_fav,Rating,Reviews,URL
0,Cozy houseboat with parking in the center of A...,Amsterdam,"$1,202",Yes,1,4.97,97322,https://www.airbnb.com/rooms/58087008775012407...
1,Classic houseboat in the city center of Amsterdam,Amsterdam,"$1,350",No,0,4.74,347,https://www.airbnb.com/rooms/22643412?search_m...
2,Beautiful stylish detached villa,Amsterdam,"$2,065",Yes,1,4.86,86242,https://www.airbnb.com/rooms/91064078589601919...
3,Spacious and comfortable cottage near Amsterdam,Amsterdam,"$3,031",Yes,1,4.95,95194,https://www.airbnb.com/rooms/51395892?search_m...
4,Numa | XL 2 Bedroom Apartment with Sofa Bed,Amsterdam,"$1,877",No,0,4.67,201,https://www.airbnb.com/rooms/11159175642092863...
...,...,...,...,...,...,...,...,...
495,Wonderful Augarten City-Apartment,Vienna,$965,No,1,4.97,97597,https://www.airbnb.com/rooms/4431063?search_mo...
496,"Urban Vienna Apartment, 6 people, 2 bathrooms",Vienna,$855,Yes,0,4.87,147,https://www.airbnb.com/rooms/10580977413369461...
497,"Naschmarkt apartment, newly refurbished, 6 people",Vienna,"$1,270",Yes,1,4.94,94242,https://www.airbnb.com/rooms/71344270011888680...
498,Fabulous City Center Apartment - Apt 18A,Vienna,"$2,020",Yes,1,4.78,78376,https://www.airbnb.com/rooms/13402481?search_m...


In [ ]:
df_scraped1.to_csv("/content/drive/MyDrive/my.flutter.2025/Instant_ai_training/airbnb_scraped_ALL_DATA.csv", index=False)

In [ ]:
df_scraped1["city"].value_counts()

,count
city,
Amsterdam,50
Athens,50
Barcelona,50
Berlin,50
Budapest,50
Lisbon,50
London,50
Paris,50
Rome,50


In [ ]:
import time
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options

# Prepare target URLs
urls_to_scrape = df_scraped1['URL'].unique().tolist()
print(f"Total unique URLs to enrich: {len(urls_to_scrape)}")

chrome_options = Options()
chrome_options.add_argument("--window-size=1920,1080")
chrome_options.add_argument("--headless")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=chrome_options)
wait = WebDriverWait(driver, 6)

enriched_details = []

try:
    for i, url in enumerate(urls_to_scrape, 1):
        print(f"Processing [{i}/{len(urls_to_scrape)}]: {url}")
        guests, bedrooms, beds, baths, host_name = "N/A", "N/A", "N/A", "N/A", "N/A"

        try:
            driver.get(url)
            # Wait for main container or body elements
            wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))
            time.sleep(1.5)

            page_text = driver.find_element(By.TAG_NAME, "body").text

            # Extract layout metrics (e.g. "4 guests · 2 bedrooms · 3 beds · 1 bath")
            # Let's search inside text
            metrics_match = re.search(r"(\d+)\s+guest[s]?.*?(\d+)\s+bedroom[s]?.*?(\d+)\s+bed[s]?.*?(\d+)\s+bath[s]?", page_text, re.IGNORECASE)
            if metrics_match:
                guests = metrics_match.group(1)
                bedrooms = metrics_match.group(2)
                beds = metrics_match.group(3)
                baths = metrics_match.group(4)
            else:
                # Individual fallbacks
                g_m = re.search(r"(\d+)\s+guest[s]?", page_text, re.IGNORECASE)
                br_m = re.search(r"(\d+)\s+bedroom[s]?", page_text, re.IGNORECASE)
                bd_m = re.search(r"(\d+)\s+bed[s]?", page_text, re.IGNORECASE)
                ba_m = re.search(r"(\d+)\s+bath[s]?", page_text, re.IGNORECASE)
                if g_m: guests = g_m.group(1)
                if br_m: bedrooms = br_m.group(1)
                if bd_m: beds = bd_m.group(1)
                if ba_m: baths = ba_m.group(1)

            # Extract Host Name
            # Typically structured as "Hosted by Name" or "Host: Name"
            host_match = re.search(r"Hosted by ([A-Z][a-z]+)", page_text)
            if host_match:
                host_name = host_match.group(1)
            else:
                host_match_alt = re.search(r"Host:\s*([A-Z][a-z]+)", page_text, re.IGNORECASE)
                if host_match_alt: host_name = host_match_alt.group(1)

        except Exception as e:
            print(f"Error processing {url}: {e}")

        enriched_details.append({
            "URL": url,
            "Guests": guests,
            "Bedrooms_Count": bedrooms,
            "Beds": beds,
            "Baths": baths,
            "Host_Name": host_name
        })

        # Output real-time feedback
        print(f"   -> G: {guests} | BR: {bedrooms} | BD: {beds} | BA: {baths} | Host: {host_name}")

        # Intermediate backup to check progress every 10 steps
        if i % 10 == 0:
            pd.DataFrame(enriched_details).to_csv("/content/enriched_backup.csv", index=False)

finally:
    driver.quit()

# Create DataFrame of scraped details
df_details = pd.DataFrame(enriched_details)
display(df_details.head())

Total unique URLs to enrich: 500
Processing [1/500]: https://www.airbnb.com/rooms/580870087750124077?search_mode=regular_search&adults=1&check_in=2027-02-08&check_out=2027-02-13&children=0&infants=0&pets=0&source_impression_id=p3_3fa66173-aeee-44e2-816e-c805a9f15f56_de9ab432-7214-4006-8b0a-c1b28bb4698b_0_580870087750124077_0&previous_page_section_name=1000&federated_search_id=3fa66173-aeee-44e2-816e-c805a9f15f56
   -> G: 4 | BR: 2 | BD: 3 | BA: 1 | Host: N/A
Processing [2/500]: https://www.airbnb.com/rooms/22643412?search_mode=regular_search&adults=1&check_in=2026-11-21&check_out=2026-11-26&children=0&infants=0&pets=0&source_impression_id=p3_1b97aaa1-8aa0-47b0-a371-8c819d74a4d7_b1c05df0-238c-4b25-8bbb-3cab46b99aae_0_22643412_0&previous_page_section_name=1000&federated_search_id=1b97aaa1-8aa0-47b0-a371-8c819d74a4d7
   -> G: 4 | BR: 2 | BD: 2 | BA: 1 | Host: Manuel
Processing [3/500]: https://www.airbnb.com/rooms/910640785896019190?search_mode=regular_search&adults=1&check_in=2026-11-08&

,URL,Guests,Bedrooms_Count,Beds,Baths,Host_Name
0,https://www.airbnb.com/rooms/58087008775012407...,4,2,3,1,N/A
1,https://www.airbnb.com/rooms/22643412?search_m...,4,2,2,1,Manuel
2,https://www.airbnb.com/rooms/91064078589601919...,6,4,5,5,Leontien
3,https://www.airbnb.com/rooms/51395892?search_m...,6,3,3,5,Mabel
4,https://www.airbnb.com/rooms/11159175642092863...,6,2,4,1,N/A


In [ ]:
# Merge details back into our original scraped dataframe
df_final_scraped = df_scraped1.merge(df_details, on="URL", how="left")

# Save to our primary workspace on GDrive
output_path = "/content/drive/MyDrive/my.flutter.2025/Instant_ai_training/airbnb_scraped_ALL_DATA_ENRICHED.csv"
df_final_scraped.to_csv(output_path, index=False)
print(f"Enriched dataset successfully saved to: {output_path}")
display(df_final_scraped.head())

Enriched dataset successfully saved to: /content/drive/MyDrive/my.flutter.2025/Instant_ai_training/airbnb_scraped_ALL_DATA_ENRICHED.csv


,Name,city,Price,Superhost,is_fav,Rating,Reviews,URL,Guests,Bedrooms_Count,Beds,Baths,Host_Name
0,Cozy houseboat with parking in the center of A...,Amsterdam,"$1,202",Yes,1,4.97,97322,https://www.airbnb.com/rooms/58087008775012407...,4,2,3,1,N/A
1,Classic houseboat in the city center of Amsterdam,Amsterdam,"$1,350",No,0,4.74,347,https://www.airbnb.com/rooms/22643412?search_m...,4,2,2,1,Manuel
2,Beautiful stylish detached villa,Amsterdam,"$2,065",Yes,1,4.86,86242,https://www.airbnb.com/rooms/91064078589601919...,6,4,5,5,Leontien
3,Spacious and comfortable cottage near Amsterdam,Amsterdam,"$3,031",Yes,1,4.95,95194,https://www.airbnb.com/rooms/51395892?search_m...,6,3,3,5,Mabel
4,Numa | XL 2 Bedroom Apartment with Sofa Bed,Amsterdam,"$1,877",No,0,4.67,201,https://www.airbnb.com/rooms/11159175642092863...,6,2,4,1,N/A


In [ ]:
with open("/content/drive/MyDrive/my.flutter.2025/Instant_ai_training/airbnb_scraped_ALL_DATA_ENRICHED.csv","r") as f:
 df_scraped1=pd.read_csv(f)

In [ ]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options

# Prepare target URLs
urls_to_scrape = df_scraped1['URL'].unique().tolist()
print(f"Total unique URLs to scrape amenities for: {len(urls_to_scrape)}")

chrome_options = Options()
chrome_options.add_argument("--window-size=1920,1080")
chrome_options.add_argument("--headless")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=chrome_options)
wait = WebDriverWait(driver, 8)

amenities_data = []

try:
    for i, url in enumerate(urls_to_scrape, 1):
        print(f"Processing [{i}/{len(urls_to_scrape)}]: {url}")
        listing_amenities = []

        try:
            driver.get(url)
            # Wait for main page to load
            wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))
            time.sleep(1.5)

            # Attempt to locate the 'Show all ... amenities' button
            # Typically contains 'Show all' and 'amenities'
            button_xpath = "//button[contains(., 'Show all') and contains(., 'amenities')]"

            try:
                # Scroll to ensure button is loaded and clickable
                btn = wait.until(EC.presence_of_element_located((By.XPATH, button_xpath)))
                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", btn)
                time.sleep(0.5)
                driver.execute_script("arguments[0].click();", btn)

                # Wait for the modal dialog containing the amenities list
                # Airbnb uses role='dialog' for popup modals
                wait.until(EC.presence_of_element_located((By.XPATH, "//div[@role='dialog']")))
                time.sleep(1.0)

                # Extract text lines inside the dialog
                dialog_element = driver.find_element(By.XPATH, "//div[@role='dialog']")
                lines = dialog_element.text.split('\n')

                # Filter out titles, categories, or structural texts to keep actual amenities
                exclude_phrases = ["What this place offers", "Close", "amenities", "Unavailable:"]
                for line in lines:
                    line_strip = line.strip()
                    if line_strip and not any(phrase in line_strip for phrase in exclude_phrases) and len(line_strip) > 2:
                        listing_amenities.append(line_strip)

            except Exception as inner_e:
                # Fallback: if modal cannot open, scrape the basic amenities visible on main page
                print(f"   (Notice: Could not open modal. Scraping visible ones...)")
                visible_elements = driver.find_elements(By.XPATH, "//div[contains(@data-section-id, 'AMENITIES_DEFAULT')]//div")
                for elem in visible_elements:
                    t = elem.text.strip()
                    if t and '\n' not in t and len(t) > 2:
                        listing_amenities.append(t)

        except Exception as e:
            print(f"   (Error loading URL: {e})")

        # Deduplicate list
        unique_amenities = list(set(listing_amenities))
        amenities_str = ", ".join(unique_amenities) if unique_amenities else "N/A"

        print(f"   -> Found {len(unique_amenities)} amenities.")
        amenities_data.append({
            "URL": url,
            "Amenities": amenities_str
        })

        # Periodic progress backup
        if i % 10 == 0:
            pd.DataFrame(amenities_data).to_csv("/content/amenities_backup.csv", index=False)

finally:
    driver.quit()

# Create DataFrame
df_amenities = pd.DataFrame(amenities_data)
display(df_amenities.head())

Total unique URLs to scrape amenities for: 500


SessionNotCreatedException: Message: session not created: Chrome instance exited. Examine ChromeDriver verbose log to determine the cause.; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
#0 0x5a22577315ea <unknown>
#1 0x5a225704b869 <unknown>
#2 0x5a225708b21d <unknown>
#3 0x5a2257086c36 <unknown>
#4 0x5a22570d7117 <unknown>
#5 0x5a22570d67dc <unknown>
#6 0x5a225709529b <unknown>
#7 0x5a22570960a1 <unknown>
#8 0x5a22576f3fec <unknown>
#9 0x5a22576f27f5 <unknown>
#10 0x5a22576dd785 <unknown>
#11 0x5a22576f345a <unknown>
#12 0x5a22576c5d99 <unknown>
#13 0x5a225771bb18 <unknown>
#14 0x5a225771bcb5 <unknown>
#15 0x5a225772ff73 <unknown>
#16 0x78d5dd8fab84 <unknown>
#17 0x78d5dd987ecc <unknown>


In [ ]:
# Merge new amenities column to our enriched dataset
if 'df_final_scraped' in globals():
    df_final_complete = df_final_scraped.merge(df_amenities, on="URL", how="left")
else:
    # Fallback to df_scraped1
    df_final_complete = df_scraped1.merge(df_amenities, on="URL", how="left")

# Save the fully loaded and updated file to GDrive
final_output_path = "/content/drive/MyDrive/my.flutter.2025/Instant_ai_training/airbnb_scraped_ALL_DATA_ENRICHED.csv"
df_final_complete.to_csv(final_output_path, index=False)
print(f"Complete enriched dataset with Amenities saved successfully to: {final_output_path}")
display(df_final_complete.head())

Complete enriched dataset with Amenities saved successfully to: /content/drive/MyDrive/my.flutter.2025/Instant_ai_training/airbnb_scraped_ALL_DATA_ENRICHED.csv


,Name,city,Price,Superhost,is_fav,Rating,Reviews,URL,Guests,Bedrooms_Count,Beds,Baths,Host_Name,Amenities
0,Cozy houseboat with parking in the center of A...,Amsterdam,"$1,202",Yes,1,4.97,97322,https://www.airbnb.com/rooms/58087008775012407...,4,2,3,1,N/A,N/A
1,Classic houseboat in the city center of Amsterdam,Amsterdam,"$1,350",No,0,4.74,347,https://www.airbnb.com/rooms/22643412?search_m...,4,2,2,1,Manuel,Prices include all fees
2,Beautiful stylish detached villa,Amsterdam,"$2,065",Yes,1,4.86,86242,https://www.airbnb.com/rooms/91064078589601919...,6,4,5,5,Leontien,Rare find! This place is usually booked
3,Spacious and comfortable cottage near Amsterdam,Amsterdam,"$3,031",Yes,1,4.95,95194,https://www.airbnb.com/rooms/51395892?search_m...,6,3,3,5,Mabel,Rare find! This place is usually booked
4,Numa | XL 2 Bedroom Apartment with Sofa Bed,Amsterdam,"$1,877",No,0,4.67,201,https://www.airbnb.com/rooms/11159175642092863...,6,2,4,1,N/A,"Bathroom, Body soap, Coffee maker, Bedroom and..."


#trial2

In [ ]:
!pip install seleniumbase pandas openpyxl --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.6/93.6 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 2.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.8/132.8 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.2/171.2 kB 7.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 701.2/701.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!google-chrome --headless --no-sandbox --disable-gpu --dump-dom https://example.com 2>&1 | head -50

/bin/bash: line 1: google-chrome: command not found


In [ ]:
!pip install --upgrade seleniumbase --quiet
!apt-get update -qq
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb || apt-get -f install -y
!apt-get install -y libnss3 libxss1 libappindicator3-1 libindicator7 --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.6/93.6 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 4.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.8/132.8 kB 9.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.2/171.2 kB 12.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 701.2/701.2 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# =====================================================
#  AIRBNB SCRAPER  —  one row per listing
#  ~50 listings × 10 cities  ⇒  ~500 rows × 30 cols
# =====================================================
import re, json, time, random, os
from datetime import datetime
from urllib.parse import urlparse, parse_qs
import pandas as pd
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from seleniumbase import Driver

# ---------------- CONFIG ----------------
cities          = ['Amsterdam','Athens','Barcelona','Berlin','Budapest',
                   'Lisbon','London','Paris','Rome','Vienna']
TARGET_PER_CITY = 50
OUTPUT_CSV      = "/content/airbnb_listings.csv"
OUTPUT_XLSX     = "/content/airbnb_listings.xlsx"

# ---------------- HELPERS ----------------
def safe_get(d, path, default=None):
    cur = d
    for p in path:
        if isinstance(cur, dict):
            cur = cur.get(p)
        elif isinstance(cur, list) and isinstance(p, int) and 0 <= p < len(cur):
            cur = cur[p]
        else:
            return default
        if cur is None:
            return default
    return cur

def get_deferred_state(driver):
    """Airbnb stores the full PDP JSON in <script id='data-deferred-state-0'>."""
    try:
        el  = driver.find_element(By.ID, "data-deferred-state-0")
        txt = el.get_attribute("textContent") or el.get_attribute("innerHTML")
        return json.loads(txt)
    except Exception:
        return None

def get_json_ld(driver):
    out = []
    try:
        for s in driver.find_elements(By.CSS_SELECTOR,
                                      'script[type="application/ld+json"]'):
            try:    out.append(json.loads(s.get_attribute("textContent")))
            except Exception: pass
    except Exception: pass
    return out

def parse_money(s):
    if s is None: return None
    s = re.sub(r"[^\d,\.]", "", str(s))
    if not s: return None
    s = s.replace(",", "")
    try:    return float(s)
    except ValueError: return None

def nights_from_url(url):
    q  = parse_qs(urlparse(url).query)
    ci, co = q.get("check_in",[None])[0], q.get("check_out",[None])[0]
    if ci and co:
        try:
            return (datetime.strptime(co, "%Y-%m-%d")
                  - datetime.strptime(ci, "%Y-%m-%d")).days
        except Exception: return None
    return None

# ---------------- CORE EXTRACTOR ----------------
def extract_listing(driver, wait, url, city):
    row = {k: None for k in [
        "Name","PropertyType","Price","Nights","PricePerNightEst",
        "Rating","Reviews","is_fav","Superhost","HostName","HostYears","HostReviews",
        "LocationName","LocationSubtitle","Latitude","Longitude",
        "Guests","Bedrooms","Beds","Baths",
        "Amenities","AmenitiesCount","Description",
        "CleaningFee","ServiceFee","Taxes",
        "AvailableDatesCount","AvailableDates","is_new"]}
    row["city"]      = city
    row["URL"]       = url
    row["is_fav"]    = 0
    row["Superhost"] = "No"
    row["is_new"]    = False

    # --- 1) JSON-LD (cheap fallback) ---
    for obj in get_json_ld(driver):
        if obj.get("@type") == "VacationRental":
            row["Name"]         = row["Name"]         or obj.get("name")
            row["Description"]  = row["Description"]  or obj.get("description")
            row["Latitude"]     = row["Latitude"]     or obj.get("latitude")
            row["Longitude"]    = row["Longitude"]    or obj.get("longitude")
            ar = obj.get("aggregateRating") or {}
            row["Rating"]       = row["Rating"]       or ar.get("ratingValue")
            row["Reviews"]      = row["Reviews"]      or ar.get("ratingCount")
            row["LocationName"] = row["LocationName"] or (obj.get("address") or {}).get("addressLocality")

    # --- 2) Deferred-state JSON (richest) ---
    state = get_deferred_state(driver)
    node  = None
    if state:
        for key, payload in (state.get("niobeClientData") or []):
            if isinstance(key, str) and key.startswith("StaysPdpSections"):
                node = safe_get(payload, ["data", "node"]); break
        if node is None:
            node = safe_get(state, ["niobeClientData", 0, 1, "data", "node"])

    if node:
        pdp = node.get("pdpPresentation") or {}

        desc = pdp.get("description") or {}
        row["Name"] = safe_get(desc, ["title","content","localizedString"]) or row["Name"]
        long_desc   = safe_get(desc, ["longDescriptionHtml","localizedString"])
        if long_desc:
            row["Description"] = re.sub(r"<[^>]+>", " ", long_desc).strip()

        ov = pdp.get("overview") or {}
        row["PropertyType"] = ov.get("title")
        for it in (ov.get("items") or []):
            itl = it.lower()
            m   = re.search(r"\d+", it)
            if not m: continue
            n = int(m.group())
            if   "guest"   in itl: row["Guests"]   = n
            elif "bedroom" in itl: row["Bedrooms"] = n
            elif "bath"    in itl: row["Baths"]    = n
            elif "bed"     in itl: row["Beds"]     = n

        loc = pdp.get("location") or {}
        row["LocationName"]     = loc.get("locationName") or row["LocationName"]
        row["LocationSubtitle"] = loc.get("subtitle")
        row["Latitude"]         = loc.get("latitude")  or row["Latitude"]
        row["Longitude"]        = loc.get("longitude") or row["Longitude"]

        amen = pdp.get("amenities") or {}
        groups = (amen.get("seeAllAmenitiesGroups")
                  or amen.get("previewAmenitiesGroups") or [])
        amens = []
        for g in groups:
            for a in (g.get("amenities") or []):
                if a.get("available") is False: continue
                t = a.get("title")
                if t and t not in amens: amens.append(t)
        if amens:
            row["Amenities"]      = " | ".join(amens)
            row["AmenitiesCount"] = len(amens)

        q  = pdp.get("quality") or {}
        rs = safe_get(q, ["listingRatingStats","overallRatingStats"]) or {}
        if rs.get("ratingAverage") is not None: row["Rating"]  = rs.get("ratingAverage")
        if rs.get("ratingCount")   is not None: row["Reviews"] = rs.get("ratingCount")
        row["is_new"] = bool(q.get("isNewListing"))
        gf = pdp.get("contextualizedGuestFavorites") or {}
        row["is_fav"] = 1 if (gf.get("isGuestFavorite")
                              or q.get("isGuestFavorite")) else 0

        host = pdp.get("hostInfo") or {}
        pd_  = host.get("passportData") or {}
        row["HostName"]  = pd_.get("name")
        row["Superhost"] = "Yes" if pd_.get("isSuperhost") else "No"
        for st in (pd_.get("stats") or []):
            if   st.get("type") == "YEARS_HOSTING": row["HostYears"]   = st.get("value")
            elif st.get("type") == "REVIEW_COUNT":  row["HostReviews"] = st.get("value")

    # --- 3) Price + nights ---
    row["Nights"] = nights_from_url(url)
    for sel in ['div[data-testid="book-it-default"]',
                'div[data-testid="book-it-default-sidebar"]',
                'div[data-testid="book-it-hover-target"]']:
        try:
            txt = driver.find_element(By.CSS_SELECTOR, sel).text
            m   = re.search(r"([\d,\.]+)\s*for\s*(\d+)\s*nights?", txt, re.IGNORECASE)
            if m:
                row["Price"]  = parse_money(m.group(1))
                row["Nights"] = int(m.group(2))
                if row["Nights"]:
                    row["PricePerNightEst"] = round(row["Price"] / row["Nights"], 2)
                break
        except Exception:
            continue

    if row["Price"] is None:
        for xp in ["//div[@data-testid='book-it-default']//span[contains(text(),'$')]",
                   "//div[@data-testid='book-it-default-sidebar']//span[contains(text(),'$')]"]:
            try:
                t  = WebDriverWait(driver, 2).until(
                        EC.presence_of_element_located((By.XPATH, xp))).text
                mm = re.search(r"[\$€£¥]\s*([\d,\.]+)", t)
                if mm:
                    row["Price"] = parse_money(mm.group(1)); break
            except Exception:
                continue

    # --- 4) Availability (inline calendar) ---
    try:
        avail = set()
        for c in driver.find_elements(By.CSS_SELECTOR,
                                      '[data-testid^="calendar-day-"]'):
            if c.get_attribute("data-is-day-blocked") == "false":
                m = re.match(r"calendar-day-(\d{2})/(\d{2})/(\d{4})",
                             c.get_attribute("data-testid") or "")
                if m:
                    mm, dd, yy = m.groups()
                    avail.add(f"{yy}-{mm}-{dd}")
        if avail:
            row["AvailableDates"]      = ",".join(sorted(avail))
            row["AvailableDatesCount"] = len(avail)
    except Exception:
        pass

    # --- 5) Cleaning / Service fee (best-effort) ---
    try:
        clicked = False
        for txt in ("Price details", "Price breakdown", "Show price details"):
            try:
                btn = driver.find_element(
                    By.XPATH,
                    f"//button[contains(., '{txt}')] | //a[contains(., '{txt}')]")
                driver.execute_script("arguments[0].click();", btn)
                clicked = True; time.sleep(1.5); break
            except Exception:
                continue
        if clicked:
            body = driver.find_element(By.TAG_NAME, "body").text
            for lbl in ("Cleaning fee","Service fee","Airbnb service fee","Taxes"):
                m = re.search(rf"{re.escape(lbl)}[^\d\n]{{0,20}}([\d,\.]+)",
                              body, re.IGNORECASE)
                if m:
                    key = lbl.replace(" ", "")
                    row[key] = parse_money(m.group(1))
            try:
                driver.find_element(By.CSS_SELECTOR,
                                    'button[aria-label="Close"]').click()
            except Exception:
                pass
    except Exception:
        pass

    if row["Rating"] is None and row["is_new"]:
        row["Rating"] = "New"
    return row

# ---------------- MAIN LOOP ----------------
all_data1 = []

for idx, city in enumerate(cities, 1):
    print(f"\n{'='*72}\n=== CITY {idx}/{len(cities)}: {city} ===\n{'='*72}")

    URL = (
        f"https://www.airbnb.com/s/{city}/homes"
        "?refinement_paths%5B%5D=%2Fhomes"
        "&date_picker_type=calendar&search_type=filter_change"
        "&flexible_trip_lengths%5B%5D=one_week"
        "&monthly_start_date=2026-02-01&monthly_length=3&monthly_end_date=2026-05-01"
        "&price_filter_input_type=2&channel=EXPLORE&price_filter_num_nights=5"
        "&min_bedrooms=2"
        "&selected_filter_order%5B%5D=min_bedrooms%3A2"
        "&selected_filter_order%5B%5D=room_types%3AEntire%20home%2Fapt"
        "&update_selected_filters=true"
        f"&query={city}&search_mode=regular_search"
        "&room_types%5B%5D=Entire%20home%2Fapt"
    )

    # SeleniumBase handles all Colab Chrome/driver pain automatically
    driver = Driver(browser="chrome", headless=True)
    driver.set_window_size(1920, 1080)
    items_scraped_this_city = 0

    try:
        driver.get(URL)
        wait = WebDriverWait(driver, 8)

        while items_scraped_this_city < TARGET_PER_CITY:
            try:
                wait.until(EC.presence_of_all_elements_located(
                    (By.CSS_SELECTOR, "div[data-testid='card-container']")))
            except Exception:
                print("  [warn] no cards found, ending this city")
                break

            listings = driver.find_elements(
                By.CSS_SELECTOR, "div[data-testid='card-container']")
            listing_urls = []
            for l in listings:
                try:
                    u = l.find_element(By.TAG_NAME, "a").get_attribute("href")
                    if u and "/rooms/" in u and u not in listing_urls:
                        listing_urls.append(u)
                except Exception:
                    continue

            print(f"  page has {len(listing_urls)} card URLs "
                  f"(city progress {items_scraped_this_city}/{TARGET_PER_CITY})")

            main_window = driver.current_window_handle

            for url in listing_urls:
                if items_scraped_this_city >= TARGET_PER_CITY:
                    break
                try:
                    driver.execute_script(f"window.open('{url}', '_blank');")
                    driver.switch_to.window(driver.window_handles[-1])
                    wait.until(EC.presence_of_element_located((By.TAG_NAME, "h1")))
                    time.sleep(1.5)     # let deferred-state JSON settle

                    row = extract_listing(driver, wait, url, city)
                    all_data1.append(row)
                    items_scraped_this_city += 1

                    print(f"  [{len(all_data1):>3}] {row['city']:<10} "
                          f"| {str(row['Price']):<10} {row['Nights']}n "
                          f"| ★{row['Rating']} ({row['Reviews']}) "
                          f"| Fav:{row['is_fav']} SH:{row['Superhost']:<3} "
                          f"| {row['AmenitiesCount']} amens "
                          f"| {row['PropertyType']}")

                    # checkpoint save every 25 rows
                    if len(all_data1) % 25 == 0:
                        pd.DataFrame(all_data1).to_csv(
                            OUTPUT_CSV, index=False, encoding="utf-8-sig")
                        print(f"        …checkpoint saved ({len(all_data1)} rows)")

                    time.sleep(random.uniform(1.0, 2.0))
                    driver.close()
                    driver.switch_to.window(main_window)

                except Exception as e:
                    print(f"  [skip] {type(e).__name__}: {str(e)[:80]}")
                    if len(driver.window_handles) > 1:
                        driver.close()
                        driver.switch_to.window(main_window)
                    continue

            # paginate to next result page
            if items_scraped_this_city < TARGET_PER_CITY:
                try:
                    next_btn = driver.find_element(
                        By.CSS_SELECTOR, "a[aria-label='Next']")
                    driver.execute_script("arguments[0].click();", next_btn)
                    time.sleep(4)
                except Exception:
                    print("  [warn] no next page — city done")
                    break

    finally:
        try:    driver.quit()
        except Exception: pass
        print(f"  → collected {items_scraped_this_city} listings in {city}")
        print(f"  → running total: {len(all_data1)}")

# ---------------- SAVE ----------------
print(f"\n{'='*72}\nDONE — {len(all_data1)} listings total\n{'='*72}")

df = pd.DataFrame(all_data1)
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Saved: {OUTPUT_CSV}")

try:
    df.to_excel(OUTPUT_XLSX, index=False)
    print(f"Saved: {OUTPUT_XLSX}")
except Exception as e:
    print(f"(xlsx skipped: {e})")

print("\nListings per city:")
print(df.groupby("city").size().to_string())

print("\nColumns:", list(df.columns))
print("\nFirst 3 rows:")
print(df.head(3).to_string())


=== CITY 1/10: Amsterdam ===
  page has 1 card URLs (city progress 0/50)
  [  1] Amsterdam  | None       5n | ★4.97 (322) | Fav:1 SH:Yes | 54 amens | Houseboat in Amsterdam, Netherlands
  page has 18 card URLs (city progress 1/50)
  [  2] Amsterdam  | None       5n | ★4.74 (1249) | Fav:0 SH:No  | 51 amens | Entire rental unit in Amsterdam, Netherlands
  [  3] Amsterdam  | None       5n | ★5 (11) | Fav:1 SH:Yes | 43 amens | Houseboat in Kortenhoef, Netherlands
  [  4] Amsterdam  | None       5n | ★3.6 (5) | Fav:0 SH:No  | 21 amens | Entire rental unit in Amsterdam, Netherlands
  [  5] Amsterdam  | None       5n | ★5 (3) | Fav:0 SH:Yes | 45 amens | Entire chalet in Vijfhuizen, Netherlands
  [  6] Amsterdam  | None       5n | ★5 (10) | Fav:1 SH:Yes | 63 amens | Entire cottage in Zaandam, Netherlands
  [  7] Amsterdam  | None       5n | ★5 (4) | Fav:0 SH:Yes | 46 amens | Entire bungalow in Vijfhuizen, Netherlands
  [  8] Amsterdam  | None       5n | ★4.73 (180) | Fav:0 SH:No  | 68 amens |

In [ ]:
# =====================================================
#  FILL PRICES IN EXISTING CSV
#  Reads airbnb_listings.csv → visits every URL →
#  extracts Price / PricePerNightEst / CleaningFee /
#  ServiceFee / Taxes → writes airbnb_listings_priced.csv
# =====================================================
import re, json, time, random
from datetime import datetime
from urllib.parse import urlparse, parse_qs

import pandas as pd
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from seleniumbase import Driver

# ---------------- CONFIG ----------------
INPUT_CSV        = "/content/airbnb_listings.csv"
OUTPUT_CSV       = "/content/airbnb_listings_priced.csv"
FORCE_REFETCH    = False          # True → re-scrape rows that already have a Price
CHECKPOINT_EVERY = 25
SLEEP_BETWEEN    = (1.5, 3.0)     # polite delay between page loads
HEADLESS         = True

# ---------------- HELPERS ----------------
def safe_get(d, path, default=None):
    cur = d
    for p in path:
        if isinstance(cur, dict):
            cur = cur.get(p)
        elif isinstance(cur, list) and isinstance(p, int) and 0 <= p < len(cur):
            cur = cur[p]
        else:
            return default
        if cur is None:
            return default
    return cur

def parse_money(s):
    if s is None: return None
    s = re.sub(r"[^\d,\.]", "", str(s))
    if not s: return None
    # If both . and , present → . is decimal, , is thousands
    if "." in s and "," in s:
        s = s.replace(",", "")
    else:
        s = s.replace(",", "")
    try:
        return float(s)
    except ValueError:
        return None

def nights_from_url(url):
    q = parse_qs(urlparse(url).query)
    ci, co = q.get("check_in", [None])[0], q.get("check_out", [None])[0]
    if ci and co:
        try:
            return (datetime.strptime(co, "%Y-%m-%d")
                    - datetime.strptime(ci, "%Y-%m-%d")).days
        except Exception:
            return None
    return None

def get_deferred_state(driver):
    try:
        el  = driver.find_element(By.ID, "data-deferred-state-0")
        txt = el.get_attribute("textContent") or el.get_attribute("innerHTML")
        return json.loads(txt)
    except Exception:
        return None

def get_json_ld(driver):
    out = []
    try:
        for s in driver.find_elements(By.CSS_SELECTOR,
                                      'script[type="application/ld+json"]'):
            try:    out.append(json.loads(s.get_attribute("textContent")))
            except Exception: pass
    except Exception: pass
    return out

# ---------------- PRICE EXTRACTORS ----------------
def price_from_ld(ld_list):
    """VacationRental JSON-LD usually carries offers[].price."""
    for obj in ld_list:
        if obj.get("@type") == "VacationRental":
            offers = obj.get("offers") or []
            if isinstance(offers, dict):
                offers = [offers]
            for o in offers:
                p = o.get("price")
                if p is not None:
                    try:
                        v = float(p)
                        if v > 0: return v
                    except Exception:
                        pass
    return None

def price_from_state(state):
    """
    Robust extraction from the data-deferred-state-0 JSON.
    Returns dict: Price, CleaningFee, ServiceFee, Taxes.
    """
    out = {"Price": None, "CleaningFee": None, "ServiceFee": None, "Taxes": None}
    if not state:
        return out

    node = None
    for key, payload in (state.get("niobeClientData") or []):
        if isinstance(key, str) and key.startswith("StaysPdpSections"):
            node = safe_get(payload, ["data", "node"]); break
    if node is None:
        node = safe_get(state, ["niobeClientData", 0, 1, "data", "node"])
    if not node:
        return out

    pdp = node.get("pdpPresentation") or {}

    # ---- 1. Structured paths (most common today) ----
    structured_paths = [
        ["structuredDisplayPrice", "primaryLine", "price"],
        ["structuredDisplayPrice", "primaryLine", "discountedPrice"],
        ["structuredDisplayPrice", "secondaryLine", "price"],
        ["bookIt", "priceBreakdown", "total"],
        ["booking", "priceBreakdown", "total"],
        ["pricing", "priceBreakdown", "total"],
        ["pricing", "primaryPrice"],
    ]
    for prefix in (node, pdp):
        for path in structured_paths:
            val = safe_get(prefix, path)
            if isinstance(val, dict):
                amt = (val.get("amount")
                       or val.get("unformattedValue")
                       or val.get("value"))
                if isinstance(amt, (int, float)) and amt > 0:
                    out["Price"] = float(amt); break
            elif isinstance(val, (int, float)) and val > 0:
                out["Price"] = float(val); break
        if out["Price"]:
            break

    # ---- 2. Recursive scan as fallback ----
    if out["Price"] is None:
        hits = []
        def walk(o, p=""):
            if isinstance(o, dict):
                for k, v in o.items():
                    kl = k.lower()
                    if kl in ("amount", "unformattedvalue", "value", "total") \
                       and isinstance(v, (int, float)) and v > 0:
                        pl = (p + "." + k).lower()
                        if any(w in pl for w in
                               ("price", "pricing", "total", "cost", "breakdown")):
                            # skip fee-like paths
                            if not any(f in pl for f in
                                       ("cleaning", "servicefee",
                                        "service_fee", "tax", "nightly",
                                        "pernight", "per_night", "monthly")):
                                hits.append((p + "." + k, v))
                    walk(v, p + "." + k)
            elif isinstance(o, list):
                for i, v in enumerate(o):
                    walk(v, f"{p}[{i}]")
        walk(node, "node")
        if hits:
            # Prefer the largest 'total' candidate
            totals = [h for h in hits if "total" in h[0].lower()]
            pool = totals or hits
            out["Price"] = max(pool, key=lambda x: x[1])[1]

    # ---- 3. Fee line items ----
    for path in (
        ["pricing", "priceBreakdown", "lineItems"],
        ["booking", "priceBreakdown", "lineItems"],
        ["bookIt", "priceBreakdown", "lineItems"],
    ):
        items = safe_get(node, path) or safe_get(pdp, path) or []
        for it in items:
            label = ((it.get("localizedTitle") or "") + " " +
                     (it.get("title") or "")).lower()
            amt = (safe_get(it, ["price", "amount"])
                   or safe_get(it, ["price", "unformattedValue"])
                   or it.get("amount"))
            if not isinstance(amt, (int, float)):
                continue
            if "clean" in label and out["CleaningFee"] is None:
                out["CleaningFee"] = float(amt)
            elif "service" in label and "fee" in label and out["ServiceFee"] is None:
                out["ServiceFee"] = float(amt)
            elif "tax" in label and out["Taxes"] is None:
                out["Taxes"] = float(amt)
        if any(out[k] is not None for k in ("CleaningFee","ServiceFee","Taxes")):
            break

    return out

def price_from_dom(driver, nights):
    """Currency-agnostic body-text fallback."""
    out = {"Price": None, "CleaningFee": None, "ServiceFee": None, "Taxes": None}
    try:
        body = driver.find_element(By.TAG_NAME, "body").text
    except Exception:
        return out

    # 1) total price
    for pat in (
        r"(?:total)\s*:?\s*([€$£¥]\s*[\d.,]+)",
        r"([€$£¥]\s*[\d.,]+)\s*(?:total|grand\s*total)",
        r"([€$£¥]\s*[\d.,]+)\s*for\s*\d+\s*nights?",
        r"for\s*\d+\s*nights?[^\n€$£¥]{0,30}([€$£¥]\s*[\d.,]+)",
    ):
        m = re.search(pat, body, re.IGNORECASE)
        if m:
            p = parse_money(m.group(1))
            if p and p > 0:
                out["Price"] = p
                break

    # 2) per-night × nights fallback
    if out["Price"] is None and nights:
        m = re.search(r"([€$£¥]\s*[\d.,]+)\s*(?:/|per\s+)night",
                      body, re.IGNORECASE)
        if m:
            pn = parse_money(m.group(1))
            if pn:
                out["Price"] = round(pn * nights, 2)

    # 3) fees
    for lbl, key in (
        ("cleaning fee",       "CleaningFee"),
        ("airbnb service fee", "ServiceFee"),
        ("service fee",        "ServiceFee"),
        ("taxes",              "Taxes"),
        ("tax",                "Taxes"),
    ):
        if out[key] is not None:
            continue
        m = re.search(rf"{re.escape(lbl)}[^\d\n€$£¥]{{0,25}}"
                      rf"([€$£¥]?\s*[\d.,]+)", body, re.IGNORECASE)
        if m:
            v = parse_money(m.group(1))
            if v: out[key] = v

    return out

# ---------------- MAIN ----------------
def has_price(row):
    p = row.get("Price")
    if p is None: return False
    try:
        if pd.isna(p): return False
        return float(p) > 0
    except Exception:
        return False

def main():
    df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")

    # Guarantee the columns exist
    for c in ("Price", "PricePerNightEst", "CleaningFee",
              "ServiceFee", "Taxes", "Nights"):
        if c not in df.columns:
            df[c] = None

    total = len(df)
    todo = [i for i, r in df.iterrows()
            if FORCE_REFETCH or not has_price(r)]
    print(f"Total rows: {total} | Rows to fetch: {len(todo)}\n")

    driver = Driver(browser="chrome", headless=HEADLESS)
    driver.set_window_size(1920, 1080)
    wait = WebDriverWait(driver, 10)

    try:
        for n, i in enumerate(todo, 1):
            url = df.at[i, "URL"]
            if not isinstance(url, str) or "airbnb" not in url:
                print(f"[{n}/{len(todo)}] row {i}: no URL, skip")
                continue

            nights = df.at[i, "Nights"]
            if pd.isna(nights) or not nights:
                nights = nights_from_url(url)
                df.at[i, "Nights"] = nights

            try:
                driver.get(url)
                wait.until(EC.presence_of_element_located((By.TAG_NAME, "h1")))
                # Let deferred-state settle
                try:
                    WebDriverWait(driver, 6).until(
                        EC.presence_of_element_located(
                            (By.ID, "data-deferred-state-0")))
                except Exception:
                    pass
                time.sleep(random.uniform(*SLEEP_BETWEEN))

                state = get_deferred_state(driver)
                ld    = get_json_ld(driver)

                data = price_from_state(state)

                # JSON-LD fallback
                if not data["Price"]:
                    p = price_from_ld(ld)
                    if p: data["Price"] = p

                # DOM fallback (also fills fees if state missed them)
                if (not data["Price"]) or any(
                    data[k] is None for k in
                    ("CleaningFee", "ServiceFee", "Taxes")):
                    dom = price_from_dom(driver, nights)
                    for k, v in dom.items():
                        if not data.get(k) and v:
                            data[k] = v

                # Write back
                df.at[i, "Price"] = data["Price"]
                df.at[i, "CleaningFee"] = data["CleaningFee"]
                df.at[i, "ServiceFee"]  = data["ServiceFee"]
                df.at[i, "Taxes"]       = data["Taxes"]
                if data["Price"] and nights:
                    df.at[i, "PricePerNightEst"] = round(
                        data["Price"] / nights, 2)

                city = df.at[i, "city"] if "city" in df.columns else "?"
                pn   = df.at[i, "PricePerNightEst"]
                print(f"[{n}/{len(todo)}] row {i:>3} | {city:<12} | "
                      f"Price={data['Price']} | {nights}n | PN={pn}")

                if n % CHECKPOINT_EVERY == 0:
                    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
                    print(f"   ↳ checkpoint saved ({n}/{len(todo)})")

            except Exception as e:
                print(f"[{n}/{len(todo)}] row {i}: "
                      f"{type(e).__name__}: {str(e)[:90]}")
                continue
    finally:
        try: driver.quit()
        except Exception: pass
        df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
        filled = df["Price"].notna().sum()
        print(f"\nDONE → {OUTPUT_CSV}")
        print(f"Rows with Price: {filled}/{total} "
              f"({filled/total*100:.1f}%)")
        print(f"Rows with CleaningFee: {df['CleaningFee'].notna().sum()}")
        print(f"Rows with ServiceFee:  {df['ServiceFee'].notna().sum()}")
        print(f"Rows with Taxes:       {df['Taxes'].notna().sum()}")

if __name__ == "__main__":
    main()

Total rows: 500 | Rows to fetch: 500

[1/500] row   0 | Amsterdam    | Price=1204.0 | 5n | PN=240.8
[2/500] row   1 | Amsterdam    | Price=None | 5n | PN=nan
[3/500] row   2 | Amsterdam    | Price=1900.0 | 5n | PN=380.0
[4/500] row   3 | Amsterdam    | Price=1497.0 | 5n | PN=299.4
[5/500] row   4 | Amsterdam    | Price=1017.0 | 5n | PN=203.4
[6/500] row   5 | Amsterdam    | Price=1310.0 | 5n | PN=262.0
[7/500] row   6 | Amsterdam    | Price=1017.0 | 5n | PN=203.4
[8/500] row   7 | Amsterdam    | Price=1052.0 | 5n | PN=210.4
[9/500] row   8 | Amsterdam    | Price=1955.0 | 5n | PN=391.0
[10/500] row   9 | Amsterdam    | Price=1874.0 | 5n | PN=374.8
[11/500] row  10 | Amsterdam    | Price=1403.0 | 5n | PN=280.6
[12/500] row  11 | Amsterdam    | Price=1744.0 | 5n | PN=348.8
[13/500] row  12 | Amsterdam    | Price=1500.0 | 5n | PN=300.0
[14/500] row  13 | Amsterdam    | Price=2352.0 | 5n | PN=470.4
[15/500] row  14 | Amsterdam    | Price=None | 5n | PN=nan
[16/500] row  15 | Amsterdam    | 

In [ ]:
# =====================================================
#  AIRBNB SCRAPER  —  one row per listing
#  ~50 listings × 10 cities  ⇒  ~500 rows × 30 cols
# =====================================================
import re, json, time, random, os
from datetime import datetime
from urllib.parse import urlparse, parse_qs
import pandas as pd
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from seleniumbase import Driver

# ---------------- CONFIG ----------------
cities          = ['Amsterdam','Athens','Barcelona','Berlin','Budapest',
                   'Lisbon','London','Paris','Rome','Vienna']
TARGET_PER_CITY = 50
OUTPUT_CSV      = "/content/airbnb_listings_weekends.csv"
OUTPUT_XLSX     = "/content/airbnb_listings_weekends.xlsx"

# ---------------- HELPERS ----------------
def safe_get(d, path, default=None):
    cur = d
    for p in path:
        if isinstance(cur, dict):
            cur = cur.get(p)
        elif isinstance(cur, list) and isinstance(p, int) and 0 <= p < len(cur):
            cur = cur[p]
        else:
            return default
        if cur is None:
            return default
    return cur

def get_deferred_state(driver):
    """Airbnb stores the full PDP JSON in <script id='data-deferred-state-0'>."""
    try:
        el  = driver.find_element(By.ID, "data-deferred-state-0")
        txt = el.get_attribute("textContent") or el.get_attribute("innerHTML")
        return json.loads(txt)
    except Exception:
        return None

def get_json_ld(driver):
    out = []
    try:
        for s in driver.find_elements(By.CSS_SELECTOR,
                                      'script[type="application/ld+json"]'):
            try:    out.append(json.loads(s.get_attribute("textContent")))
            except Exception: pass
    except Exception: pass
    return out

def parse_money(s):
    if s is None: return None
    s = re.sub(r"[^\d,\.]", "", str(s))
    if not s: return None
    s = s.replace(",", "")
    try:    return float(s)
    except ValueError: return None

def nights_from_url(url):
    q  = parse_qs(urlparse(url).query)
    ci, co = q.get("check_in",[None])[0], q.get("check_out",[None])[0]
    if ci and co:
        try:
            return (datetime.strptime(co, "%Y-%m-%d")
                  - datetime.strptime(ci, "%Y-%m-%d")).days
        except Exception: return None
    return None

# ---------------- CORE EXTRACTOR ----------------
def extract_listing(driver, wait, url, city):
    row = {k: None for k in [
        "Name","PropertyType","Price","Nights","PricePerNightEst",
        "Rating","Reviews","is_fav","Superhost","HostName","HostYears","HostReviews",
        "LocationName","LocationSubtitle","Latitude","Longitude",
        "Guests","Bedrooms","Beds","Baths",
        "Amenities","AmenitiesCount","Description",
        "CleaningFee","ServiceFee","Taxes",
        "AvailableDatesCount","AvailableDates","is_new"]}
    row["city"]      = city
    row["URL"]       = url
    row["is_fav"]    = 0
    row["Superhost"] = "No"
    row["is_new"]    = False

    # --- 1) JSON-LD (cheap fallback) ---
    for obj in get_json_ld(driver):
        if obj.get("@type") == "VacationRental":
            row["Name"]         = row["Name"]         or obj.get("name")
            row["Description"]  = row["Description"]  or obj.get("description")
            row["Latitude"]     = row["Latitude"]     or obj.get("latitude")
            row["Longitude"]    = row["Longitude"]    or obj.get("longitude")
            ar = obj.get("aggregateRating") or {}
            row["Rating"]       = row["Rating"]       or ar.get("ratingValue")
            row["Reviews"]      = row["Reviews"]      or ar.get("ratingCount")
            row["LocationName"] = row["LocationName"] or (obj.get("address") or {}).get("addressLocality")

    # --- 2) Deferred-state JSON (richest) ---
    state = get_deferred_state(driver)
    node  = None
    if state:
        for key, payload in (state.get("niobeClientData") or []):
            if isinstance(key, str) and key.startswith("StaysPdpSections"):
                node = safe_get(payload, ["data", "node"]); break
        if node is None:
            node = safe_get(state, ["niobeClientData", 0, 1, "data", "node"])

    if node:
        pdp = node.get("pdpPresentation") or {}

        desc = pdp.get("description") or {}
        row["Name"] = safe_get(desc, ["title","content","localizedString"]) or row["Name"]
        long_desc   = safe_get(desc, ["longDescriptionHtml","localizedString"])
        if long_desc:
            row["Description"] = re.sub(r"<[^>]+>", " ", long_desc).strip()

        ov = pdp.get("overview") or {}
        row["PropertyType"] = ov.get("title")
        for it in (ov.get("items") or []):
            itl = it.lower()
            m   = re.search(r"\d+", it)
            if not m: continue
            n = int(m.group())
            if   "guest"   in itl: row["Guests"]   = n
            elif "bedroom" in itl: row["Bedrooms"] = n
            elif "bath"    in itl: row["Baths"]    = n
            elif "bed"     in itl: row["Beds"]     = n

        loc = pdp.get("location") or {}
        row["LocationName"]     = loc.get("locationName") or row["LocationName"]
        row["LocationSubtitle"] = loc.get("subtitle")
        row["Latitude"]         = loc.get("latitude")  or row["Latitude"]
        row["Longitude"]        = loc.get("longitude") or row["Longitude"]

        amen = pdp.get("amenities") or {}
        groups = (amen.get("seeAllAmenitiesGroups")
                  or amen.get("previewAmenitiesGroups") or [])
        amens = []
        for g in groups:
            for a in (g.get("amenities") or []):
                if a.get("available") is False: continue
                t = a.get("title")
                if t and t not in amens: amens.append(t)
        if amens:
            row["Amenities"]      = " | ".join(amens)
            row["AmenitiesCount"] = len(amens)

        q  = pdp.get("quality") or {}
        rs = safe_get(q, ["listingRatingStats","overallRatingStats"]) or {}
        if rs.get("ratingAverage") is not None: row["Rating"]  = rs.get("ratingAverage")
        if rs.get("ratingCount")   is not None: row["Reviews"] = rs.get("ratingCount")
        row["is_new"] = bool(q.get("isNewListing"))
        gf = pdp.get("contextualizedGuestFavorites") or {}
        row["is_fav"] = 1 if (gf.get("isGuestFavorite")
                              or q.get("isGuestFavorite")) else 0

        host = pdp.get("hostInfo") or {}
        pd_  = host.get("passportData") or {}
        row["HostName"]  = pd_.get("name")
        row["Superhost"] = "Yes" if pd_.get("isSuperhost") else "No"
        for st in (pd_.get("stats") or []):
            if   st.get("type") == "YEARS_HOSTING": row["HostYears"]   = st.get("value")
            elif st.get("type") == "REVIEW_COUNT":  row["HostReviews"] = st.get("value")

    # --- 3) Price + nights ---
    row["Nights"] = nights_from_url(url)
    for sel in ['div[data-testid="book-it-default"]',
                'div[data-testid="book-it-default-sidebar"]',
                'div[data-testid="book-it-hover-target"]']:
        try:
            txt = driver.find_element(By.CSS_SELECTOR, sel).text
            m   = re.search(r"([\d,\.]+)\s*for\s*(\d+)\s*nights?", txt, re.IGNORECASE)
            if m:
                row["Price"]  = parse_money(m.group(1))
                row["Nights"] = int(m.group(2))
                if row["Nights"]:
                    row["PricePerNightEst"] = round(row["Price"] / row["Nights"], 2)
                break
        except Exception:
            continue

    if row["Price"] is None:
        for xp in ["//div[@data-testid='book-it-default']//span[contains(text(),'$')]",
                   "//div[@data-testid='book-it-default-sidebar']//span[contains(text(),'$')]"]:
            try:
                t  = WebDriverWait(driver, 2).until(
                        EC.presence_of_element_located((By.XPATH, xp))).text
                mm = re.search(r"[\$€£¥]\s*([\d,\.]+)", t)
                if mm:
                    row["Price"] = parse_money(mm.group(1)); break
            except Exception:
                continue

    # --- 4) Availability (inline calendar) ---
    try:
        avail = set()
        for c in driver.find_elements(By.CSS_SELECTOR,
                                      '[data-testid^="calendar-day-"]'):
            if c.get_attribute("data-is-day-blocked") == "false":
                m = re.match(r"calendar-day-(\d{2})/(\d{2})/(\d{4})",
                             c.get_attribute("data-testid") or "")
                if m:
                    mm, dd, yy = m.groups()
                    avail.add(f"{yy}-{mm}-{dd}")
        if avail:
            row["AvailableDates"]      = ",".join(sorted(avail))
            row["AvailableDatesCount"] = len(avail)
    except Exception:
        pass

    # --- 5) Cleaning / Service fee (best-effort) ---
    try:
        clicked = False
        for txt in ("Price details", "Price breakdown", "Show price details"):
            try:
                btn = driver.find_element(
                    By.XPATH,
                    f"//button[contains(., '{txt}')] | //a[contains(., '{txt}')]")
                driver.execute_script("arguments[0].click();", btn)
                clicked = True; time.sleep(1.5); break
            except Exception:
                continue
        if clicked:
            body = driver.find_element(By.TAG_NAME, "body").text
            for lbl in ("Cleaning fee","Service fee","Airbnb service fee","Taxes"):
                m = re.search(rf"{re.escape(lbl)}[^\d\n]{{0,20}}([\d,\.]+)",
                              body, re.IGNORECASE)
                if m:
                    key = lbl.replace(" ", "")
                    row[key] = parse_money(m.group(1))
            try:
                driver.find_element(By.CSS_SELECTOR,
                                    'button[aria-label="Close"]').click()
            except Exception:
                pass
    except Exception:
        pass

    if row["Rating"] is None and row["is_new"]:
        row["Rating"] = "New"
    return row

# ---------------- MAIN LOOP ----------------
all_data1 = []

for idx, city in enumerate(cities, 1):
    print(f"\n{'='*72}\n=== CITY {idx}/{len(cities)}: {city} ===\n{'='*72}")

    URL = (
        f"https://www.airbnb.com/s/{city}/homes"
        "?refinement_paths%5B%5D=%2Fhomes"
        "&date_picker_type=calendar&search_type=filter_change"
        "&flexible_trip_lengths%5B%5D=weekend_trip"
        "&monthly_start_date=2026-02-01&monthly_length=3&monthly_end_date=2026-05-01"
        "&price_filter_input_type=2&channel=EXPLORE&price_filter_num_nights=5"
        "&min_bedrooms=2"
        "&selected_filter_order%5B%5D=min_bedrooms%3A2"
        "&selected_filter_order%5B%5D=room_types%3AEntire%20home%2Fapt"
        "&update_selected_filters=true"
        f"&query={city}&search_mode=regular_search"
        "&room_types%5B%5D=Entire%20home%2Fapt"
    )

    # SeleniumBase handles all Colab Chrome/driver pain automatically
    driver = Driver(browser="chrome", headless=True)
    driver.set_window_size(1920, 1080)
    items_scraped_this_city = 0

    try:
        driver.get(URL)
        wait = WebDriverWait(driver, 8)

        while items_scraped_this_city < TARGET_PER_CITY:
            try:
                wait.until(EC.presence_of_all_elements_located(
                    (By.CSS_SELECTOR, "div[data-testid='card-container']")))
            except Exception:
                print("  [warn] no cards found, ending this city")
                break

            listings = driver.find_elements(
                By.CSS_SELECTOR, "div[data-testid='card-container']")
            listing_urls = []
            for l in listings:
                try:
                    u = l.find_element(By.TAG_NAME, "a").get_attribute("href")
                    if u and "/rooms/" in u and u not in listing_urls:
                        listing_urls.append(u)
                except Exception:
                    continue

            print(f"  page has {len(listing_urls)} card URLs "
                  f"(city progress {items_scraped_this_city}/{TARGET_PER_CITY})")

            main_window = driver.current_window_handle

            for url in listing_urls:
                if items_scraped_this_city >= TARGET_PER_CITY:
                    break
                try:
                    driver.execute_script(f"window.open('{url}', '_blank');")
                    driver.switch_to.window(driver.window_handles[-1])
                    wait.until(EC.presence_of_element_located((By.TAG_NAME, "h1")))
                    time.sleep(1.5)     # let deferred-state JSON settle

                    row = extract_listing(driver, wait, url, city)
                    all_data1.append(row)
                    items_scraped_this_city += 1

                    print(f"  [{len(all_data1):>3}] {row['city']:<10} "
                          f"| {str(row['Price']):<10} {row['Nights']}n "
                          f"| ★{row['Rating']} ({row['Reviews']}) "
                          f"| Fav:{row['is_fav']} SH:{row['Superhost']:<3} "
                          f"| {row['AmenitiesCount']} amens "
                          f"| {row['PropertyType']}")

                    # checkpoint save every 25 rows
                    if len(all_data1) % 25 == 0:
                        pd.DataFrame(all_data1).to_csv(
                            OUTPUT_CSV, index=False, encoding="utf-8-sig")
                        print(f"        …checkpoint saved ({len(all_data1)} rows)")

                    time.sleep(random.uniform(1.0, 2.0))
                    driver.close()
                    driver.switch_to.window(main_window)

                except Exception as e:
                    print(f"  [skip] {type(e).__name__}: {str(e)[:80]}")
                    if len(driver.window_handles) > 1:
                        driver.close()
                        driver.switch_to.window(main_window)
                    continue

            # paginate to next result page
            if items_scraped_this_city < TARGET_PER_CITY:
                try:
                    next_btn = driver.find_element(
                        By.CSS_SELECTOR, "a[aria-label='Next']")
                    driver.execute_script("arguments[0].click();", next_btn)
                    time.sleep(4)
                except Exception:
                    print("  [warn] no next page — city done")
                    break

    finally:
        try:    driver.quit()
        except Exception: pass
        print(f"  → collected {items_scraped_this_city} listings in {city}")
        print(f"  → running total: {len(all_data1)}")

# ---------------- SAVE ----------------
print(f"\n{'='*72}\nDONE — {len(all_data1)} listings total\n{'='*72}")

df = pd.DataFrame(all_data1)
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Saved: {OUTPUT_CSV}")

try:
    df.to_excel(OUTPUT_XLSX, index=False)
    print(f"Saved: {OUTPUT_XLSX}")
except Exception as e:
    print(f"(xlsx skipped: {e})")

print("\nListings per city:")
print(df.groupby("city").size().to_string())

print("\nColumns:", list(df.columns))
print("\nFirst 3 rows:")
print(df.head(3).to_string())


=== CITY 1/10: Amsterdam ===
  page has 1 card URLs (city progress 0/50)
  [  1] Amsterdam  | None       2n | ★4.67 (3) | Fav:0 SH:Yes | 45 amens | Entire bungalow in Vijfhuizen, Netherlands
  page has 18 card URLs (city progress 1/50)
  [  2] Amsterdam  | None       2n | ★4.85 (763) | Fav:1 SH:Yes | 47 amens | Entire serviced apartment in Amsterdam, Netherlands
  [  3] Amsterdam  | None       2n | ★4.35 (26) | Fav:0 SH:No  | 45 amens | Entire serviced apartment in Amsterdam, Netherlands
  [  4] Amsterdam  | None       2n | ★4.86 (310) | Fav:1 SH:Yes | 39 amens | Entire rental unit in Amsterdam, Netherlands
  [  5] Amsterdam  | None       2n | ★5 (3) | Fav:0 SH:Yes | 45 amens | Entire chalet in Vijfhuizen, Netherlands
  [  6] Amsterdam  | None       2n | ★4.7 (325) | Fav:0 SH:No  | 43 amens | Entire serviced apartment in Amsterdam, Netherlands
  [  7] Amsterdam  | None       2n | ★4.99 (90) | Fav:1 SH:Yes | 34 amens | Houseboat in Amsterdam, Netherlands
  [  8] Amsterdam  | None      

In [ ]:
# =====================================================
#  FILL PRICES IN EXISTING CSV
#  Reads airbnb_listings.csv → visits every URL →
#  extracts Price / PricePerNightEst / CleaningFee /
#  ServiceFee / Taxes → writes airbnb_listings_priced.csv
# =====================================================
import re, json, time, random
from datetime import datetime
from urllib.parse import urlparse, parse_qs

import pandas as pd
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from seleniumbase import Driver

# ---------------- CONFIG ----------------
INPUT_CSV        = "/content/airbnb_listings_weekends.csv"
OUTPUT_CSV       = "/content/airbnb_listings_weekends_priced.csv"
FORCE_REFETCH    = False          # True → re-scrape rows that already have a Price
CHECKPOINT_EVERY = 25
SLEEP_BETWEEN    = (1.5, 3.0)     # polite delay between page loads
HEADLESS         = True

# ---------------- HELPERS ----------------
def safe_get(d, path, default=None):
    cur = d
    for p in path:
        if isinstance(cur, dict):
            cur = cur.get(p)
        elif isinstance(cur, list) and isinstance(p, int) and 0 <= p < len(cur):
            cur = cur[p]
        else:
            return default
        if cur is None:
            return default
    return cur

def parse_money(s):
    if s is None: return None
    s = re.sub(r"[^\d,\.]", "", str(s))
    if not s: return None
    # If both . and , present → . is decimal, , is thousands
    if "." in s and "," in s:
        s = s.replace(",", "")
    else:
        s = s.replace(",", "")
    try:
        return float(s)
    except ValueError:
        return None

def nights_from_url(url):
    q = parse_qs(urlparse(url).query)
    ci, co = q.get("check_in", [None])[0], q.get("check_out", [None])[0]
    if ci and co:
        try:
            return (datetime.strptime(co, "%Y-%m-%d")
                    - datetime.strptime(ci, "%Y-%m-%d")).days
        except Exception:
            return None
    return None

def get_deferred_state(driver):
    try:
        el  = driver.find_element(By.ID, "data-deferred-state-0")
        txt = el.get_attribute("textContent") or el.get_attribute("innerHTML")
        return json.loads(txt)
    except Exception:
        return None

def get_json_ld(driver):
    out = []
    try:
        for s in driver.find_elements(By.CSS_SELECTOR,
                                      'script[type="application/ld+json"]'):
            try:    out.append(json.loads(s.get_attribute("textContent")))
            except Exception: pass
    except Exception: pass
    return out

# ---------------- PRICE EXTRACTORS ----------------
def price_from_ld(ld_list):
    """VacationRental JSON-LD usually carries offers[].price."""
    for obj in ld_list:
        if obj.get("@type") == "VacationRental":
            offers = obj.get("offers") or []
            if isinstance(offers, dict):
                offers = [offers]
            for o in offers:
                p = o.get("price")
                if p is not None:
                    try:
                        v = float(p)
                        if v > 0: return v
                    except Exception:
                        pass
    return None

def price_from_state(state):
    """
    Robust extraction from the data-deferred-state-0 JSON.
    Returns dict: Price, CleaningFee, ServiceFee, Taxes.
    """
    out = {"Price": None, "CleaningFee": None, "ServiceFee": None, "Taxes": None}
    if not state:
        return out

    node = None
    for key, payload in (state.get("niobeClientData") or []):
        if isinstance(key, str) and key.startswith("StaysPdpSections"):
            node = safe_get(payload, ["data", "node"]); break
    if node is None:
        node = safe_get(state, ["niobeClientData", 0, 1, "data", "node"])
    if not node:
        return out

    pdp = node.get("pdpPresentation") or {}

    # ---- 1. Structured paths (most common today) ----
    structured_paths = [
        ["structuredDisplayPrice", "primaryLine", "price"],
        ["structuredDisplayPrice", "primaryLine", "discountedPrice"],
        ["structuredDisplayPrice", "secondaryLine", "price"],
        ["bookIt", "priceBreakdown", "total"],
        ["booking", "priceBreakdown", "total"],
        ["pricing", "priceBreakdown", "total"],
        ["pricing", "primaryPrice"],
    ]
    for prefix in (node, pdp):
        for path in structured_paths:
            val = safe_get(prefix, path)
            if isinstance(val, dict):
                amt = (val.get("amount")
                       or val.get("unformattedValue")
                       or val.get("value"))
                if isinstance(amt, (int, float)) and amt > 0:
                    out["Price"] = float(amt); break
            elif isinstance(val, (int, float)) and val > 0:
                out["Price"] = float(val); break
        if out["Price"]:
            break

    # ---- 2. Recursive scan as fallback ----
    if out["Price"] is None:
        hits = []
        def walk(o, p=""):
            if isinstance(o, dict):
                for k, v in o.items():
                    kl = k.lower()
                    if kl in ("amount", "unformattedvalue", "value", "total") \
                       and isinstance(v, (int, float)) and v > 0:
                        pl = (p + "." + k).lower()
                        if any(w in pl for w in
                               ("price", "pricing", "total", "cost", "breakdown")):
                            # skip fee-like paths
                            if not any(f in pl for f in
                                       ("cleaning", "servicefee",
                                        "service_fee", "tax", "nightly",
                                        "pernight", "per_night", "monthly")):
                                hits.append((p + "." + k, v))
                    walk(v, p + "." + k)
            elif isinstance(o, list):
                for i, v in enumerate(o):
                    walk(v, f"{p}[{i}]")
        walk(node, "node")
        if hits:
            # Prefer the largest 'total' candidate
            totals = [h for h in hits if "total" in h[0].lower()]
            pool = totals or hits
            out["Price"] = max(pool, key=lambda x: x[1])[1]

    # ---- 3. Fee line items ----
    for path in (
        ["pricing", "priceBreakdown", "lineItems"],
        ["booking", "priceBreakdown", "lineItems"],
        ["bookIt", "priceBreakdown", "lineItems"],
    ):
        items = safe_get(node, path) or safe_get(pdp, path) or []
        for it in items:
            label = ((it.get("localizedTitle") or "") + " " +
                     (it.get("title") or "")).lower()
            amt = (safe_get(it, ["price", "amount"])
                   or safe_get(it, ["price", "unformattedValue"])
                   or it.get("amount"))
            if not isinstance(amt, (int, float)):
                continue
            if "clean" in label and out["CleaningFee"] is None:
                out["CleaningFee"] = float(amt)
            elif "service" in label and "fee" in label and out["ServiceFee"] is None:
                out["ServiceFee"] = float(amt)
            elif "tax" in label and out["Taxes"] is None:
                out["Taxes"] = float(amt)
        if any(out[k] is not None for k in ("CleaningFee","ServiceFee","Taxes")):
            break

    return out

def price_from_dom(driver, nights):
    """Currency-agnostic body-text fallback."""
    out = {"Price": None, "CleaningFee": None, "ServiceFee": None, "Taxes": None}
    try:
        body = driver.find_element(By.TAG_NAME, "body").text
    except Exception:
        return out

    # 1) total price
    for pat in (
        r"(?:total)\s*:?\s*([€$£¥]\s*[\d.,]+)",
        r"([€$£¥]\s*[\d.,]+)\s*(?:total|grand\s*total)",
        r"([€$£¥]\s*[\d.,]+)\s*for\s*\d+\s*nights?",
        r"for\s*\d+\s*nights?[^\n€$£¥]{0,30}([€$£¥]\s*[\d.,]+)",
    ):
        m = re.search(pat, body, re.IGNORECASE)
        if m:
            p = parse_money(m.group(1))
            if p and p > 0:
                out["Price"] = p
                break

    # 2) per-night × nights fallback
    if out["Price"] is None and nights:
        m = re.search(r"([€$£¥]\s*[\d.,]+)\s*(?:/|per\s+)night",
                      body, re.IGNORECASE)
        if m:
            pn = parse_money(m.group(1))
            if pn:
                out["Price"] = round(pn * nights, 2)

    # 3) fees
    for lbl, key in (
        ("cleaning fee",       "CleaningFee"),
        ("airbnb service fee", "ServiceFee"),
        ("service fee",        "ServiceFee"),
        ("taxes",              "Taxes"),
        ("tax",                "Taxes"),
    ):
        if out[key] is not None:
            continue
        m = re.search(rf"{re.escape(lbl)}[^\d\n€$£¥]{{0,25}}"
                      rf"([€$£¥]?\s*[\d.,]+)", body, re.IGNORECASE)
        if m:
            v = parse_money(m.group(1))
            if v: out[key] = v

    return out

# ---------------- MAIN ----------------
def has_price(row):
    p = row.get("Price")
    if p is None: return False
    try:
        if pd.isna(p): return False
        return float(p) > 0
    except Exception:
        return False

def main():
    df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")

    # Guarantee the columns exist
    for c in ("Price", "PricePerNightEst", "CleaningFee",
              "ServiceFee", "Taxes", "Nights"):
        if c not in df.columns:
            df[c] = None

    total = len(df)
    todo = [i for i, r in df.iterrows()
            if FORCE_REFETCH or not has_price(r)]
    print(f"Total rows: {total} | Rows to fetch: {len(todo)}\n")

    driver = Driver(browser="chrome", headless=HEADLESS)
    driver.set_window_size(1920, 1080)
    wait = WebDriverWait(driver, 10)

    try:
        for n, i in enumerate(todo, 1):
            url = df.at[i, "URL"]
            if not isinstance(url, str) or "airbnb" not in url:
                print(f"[{n}/{len(todo)}] row {i}: no URL, skip")
                continue

            nights = df.at[i, "Nights"]
            if pd.isna(nights) or not nights:
                nights = nights_from_url(url)
                df.at[i, "Nights"] = nights

            try:
                driver.get(url)
                wait.until(EC.presence_of_element_located((By.TAG_NAME, "h1")))
                # Let deferred-state settle
                try:
                    WebDriverWait(driver, 6).until(
                        EC.presence_of_element_located(
                            (By.ID, "data-deferred-state-0")))
                except Exception:
                    pass
                time.sleep(random.uniform(*SLEEP_BETWEEN))

                state = get_deferred_state(driver)
                ld    = get_json_ld(driver)

                data = price_from_state(state)

                # JSON-LD fallback
                if not data["Price"]:
                    p = price_from_ld(ld)
                    if p: data["Price"] = p

                # DOM fallback (also fills fees if state missed them)
                if (not data["Price"]) or any(
                    data[k] is None for k in
                    ("CleaningFee", "ServiceFee", "Taxes")):
                    dom = price_from_dom(driver, nights)
                    for k, v in dom.items():
                        if not data.get(k) and v:
                            data[k] = v

                # Write back
                df.at[i, "Price"] = data["Price"]
                df.at[i, "CleaningFee"] = data["CleaningFee"]
                df.at[i, "ServiceFee"]  = data["ServiceFee"]
                df.at[i, "Taxes"]       = data["Taxes"]
                if data["Price"] and nights:
                    df.at[i, "PricePerNightEst"] = round(
                        data["Price"] / nights, 2)

                city = df.at[i, "city"] if "city" in df.columns else "?"
                pn   = df.at[i, "PricePerNightEst"]
                print(f"[{n}/{len(todo)}] row {i:>3} | {city:<12} | "
                      f"Price={data['Price']} | {nights}n | PN={pn}")

                if n % CHECKPOINT_EVERY == 0:
                    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
                    print(f"   ↳ checkpoint saved ({n}/{len(todo)})")

            except Exception as e:
                print(f"[{n}/{len(todo)}] row {i}: "
                      f"{type(e).__name__}: {str(e)[:90]}")
                continue
    finally:
        try: driver.quit()
        except Exception: pass
        df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
        filled = df["Price"].notna().sum()
        print(f"\nDONE → {OUTPUT_CSV}")
        print(f"Rows with Price: {filled}/{total} "
              f"({filled/total*100:.1f}%)")
        print(f"Rows with CleaningFee: {df['CleaningFee'].notna().sum()}")
        print(f"Rows with ServiceFee:  {df['ServiceFee'].notna().sum()}")
        print(f"Rows with Taxes:       {df['Taxes'].notna().sum()}")

if __name__ == "__main__":
    main()

Total rows: 500 | Rows to fetch: 500

[1/500] row   0 | Amsterdam    | Price=548.0 | 2n | PN=274.0
[2/500] row   1 | Amsterdam    | Price=1902.0 | 2n | PN=951.0
[3/500] row   2 | Amsterdam    | Price=770.0 | 2n | PN=385.0
[4/500] row   3 | Amsterdam    | Price=669.0 | 2n | PN=334.5
[5/500] row   4 | Amsterdam    | Price=548.0 | 2n | PN=274.0
[6/500] row   5 | Amsterdam    | Price=None | 2n | PN=nan
[7/500] row   6 | Amsterdam    | Price=1241.0 | 2n | PN=620.5
[8/500] row   7 | Amsterdam    | Price=771.0 | 2n | PN=385.5
[9/500] row   8 | Amsterdam    | Price=923.0 | 2n | PN=461.5
[10/500] row   9 | Amsterdam    | Price=1109.0 | 2n | PN=554.5
[11/500] row  10 | Amsterdam    | Price=642.0 | 2n | PN=321.0
[12/500] row  11 | Amsterdam    | Price=315.0 | 2n | PN=157.5
[13/500] row  12 | Amsterdam    | Price=1335.0 | 2n | PN=667.5
[14/500] row  13 | Amsterdam    | Price=240.0 | 2n | PN=120.0
[15/500] row  14 | Amsterdam    | Price=967.0 | 2n | PN=483.5
[16/500] row  15 | Amsterdam    | Price=

In [ ]:
# =====================================================
#  MERGED PRICED CSV → ADD GEOPGRAPHIC + RATING FEATURES
#  Adds: biz, guest_satisfaction_overall, dist, metro_dist,
#        attr_index, attr_index_norm, rest_index, rest_index_norm
#  Reads priced CSV in place, writes back (or to _enriched)
# =====================================================
import re, json, time, math, random, os
from pathlib import Path
from urllib.parse import urlparse, parse_qs
from datetime import datetime

import pandas as pd
import requests
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from seleniumbase import Driver

# ---------------- CONFIG ----------------
INPUT_CSV         = "/content/drive/MyDrive/my.flutter.2025/Instant_ai_training/SPRINT_01/airbnb_listings_weekdays_priced.csv"     # priced, ready
OUTPUT_CSV        = "/content/drive/MyDrive/my.flutter.2025/Instant_ai_training/SPRINT_01/airbnb_listings_weekdays_priced_enriched.csv"
OVERPASS_CACHE    = "/content/overpass_cache.json"
POI_RADIUS_KM     = 1.5
SLEEP_BETWEEN     = (1.5, 3.0)     # between URL loads
OVERPASS_SLEEP    = 1.2            # between Overpass requests
CHECKPOINT_EVERY  = 25
HEADLESS          = True
FORCE_REFRESH     = False          # True → re-scrape even if column already has value

OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.openstreetmap.ru/api/interpreter",
]

CITY_CENTRES = {
    "Amsterdam": (52.3676, 4.9041),
    "Athens":    (37.9838, 23.7275),
    "Barcelona": (41.3874, 2.1686),
    "Berlin":    (52.5200, 13.4050),
    "Budapest":  (47.4979, 19.0402),
    "Lisbon":    (38.7223, -9.1393),
    "London":    (51.5074, -0.1278),
    "Paris":     (48.8566, 2.3522),
    "Rome":      (41.9028, 12.4964),
    "Vienna":    (48.2082, 16.3738),
}

TARGET_COLS = [
    "biz", "guest_satisfaction_overall",
    "dist", "metro_dist",
    "attr_index", "attr_index_norm",
    "rest_index", "rest_index_norm",
]

# ---------------- HELPERS ----------------
def safe_get(d, path, default=None):
    cur = d
    for p in path:
        if isinstance(cur, dict):
            cur = cur.get(p)
        elif isinstance(cur, list) and isinstance(p, int) and 0 <= p < len(cur):
            cur = cur[p]
        else:
            return default
        if cur is None:
            return default
    return cur

def get_json_ld(driver):
    out = []
    try:
        for s in driver.find_elements(By.CSS_SELECTOR, 'script[type="application/ld+json"]'):
            try:
                out.append(json.loads(s.get_attribute("textContent")))
            except Exception:
                pass
    except Exception:
        pass
    return out

def rating_from_ld(ld_list):
    """Extract ratingValue (0-5) from JSON-LD aggregateRating."""
    for obj in ld_list:
        if not isinstance(obj, dict):
            continue
        agg = obj.get("aggregateRating")
        if isinstance(agg, dict):
            rv = agg.get("ratingValue")
            if rv is not None:
                try:
                    v = float(rv)
                    if v > 0:
                        return v
                except Exception:
                    pass
    return None

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(a))

# ---- Overpass with cache ----
def load_cache():
    if os.path.exists(OVERPASS_CACHE):
        try:
            with open(OVERPASS_CACHE) as f:
                return json.load(f)
        except Exception:
            return {}
    return {}

def save_cache(c):
    try:
        with open(OVERPASS_CACHE, "w") as f:
            json.dump(c, f)
    except Exception:
        pass

OVERPASS_CACHE_DATA = load_cache()

def _cache_key(lat, lon, kind):
    # round to ~11m precision to allow reuse
    return f"{round(lat, 4)}_{round(lon, 4)}_{kind}"

def _overpass(query, retries=3):
    for url in OVERPASS_ENDPOINTS:
        for attempt in range(retries):
            try:
                r = requests.post(url, data={"data": query}, timeout=60)
                if r.status_code == 200:
                    return r.json()
            except Exception:
                pass
            time.sleep(2 ** attempt)
    return None

def _count_pois(lat, lon, radius_km, tags_block):
    r = int(radius_km * 1000)
    q = f"""
    [out:json][timeout:25];
    (
      node{tags_block}(around:{r},{lat},{lon});
      way{tags_block}(around:{r},{lat},{lon});
    );
    out count;
    """
    data = _overpass(q)
    if not data:
        return None
    try:
        return int(data["elements"][0]["tags"]["total"])
    except Exception:
        return 0

def metro_dist_km(lat, lon, max_radius_km=5.0):
    r = int(max_radius_km * 1000)
    q = f"""
    [out:json][timeout:25];
    (
      node["railway"="station"]["station"="subway"](around:{r},{lat},{lon});
      node["railway"="subway_entrance"](around:{r},{lat},{lon});
      node["public_transport"="station"]["subway"="yes"](around:{r},{lat},{lon});
    );
    out body;
    """
    data = _overpass(q)
    if not data or not data.get("elements"):
        return None
    return round(min(haversine_km(lat, lon, el["lat"], el["lon"])
                     for el in data["elements"]), 3)

def get_geo_features(lat, lon):
    """Returns dict with metro_dist, attr_index, rest_index (with caching)."""
    out = {"metro_dist": None, "attr_index": None, "rest_index": None}
    for kind, fn in (
        ("metro", lambda: metro_dist_km(lat, lon)),
        ("attr",  lambda: _count_pois(lat, lon, POI_RADIUS_KM,
                     '["tourism"~"attraction|museum|viewpoint|artwork|gallery"]')),
        ("rest",  lambda: _count_pois(lat, lon, POI_RADIUS_KM,
                     '["amenity"~"restaurant|cafe|bar|fast_food|pub"]')),
    ):
        key = _cache_key(lat, lon, kind)
        if key in OVERPASS_CACHE_DATA:
            val = OVERPASS_CACHE_DATA[key]
        else:
            val = fn()
            OVERPASS_CACHE_DATA[key] = val
            time.sleep(OVERPASS_SLEEP)
        if kind == "metro":
            out["metro_dist"] = val
        elif kind == "attr":
            out["attr_index"] = val
        else:
            out["rest_index"] = val
    return out

def minmax(s):
    s = pd.to_numeric(s, errors="coerce")
    lo, hi = s.min(), s.max()
    if pd.isna(lo) or hi == lo:
        return s * 0
    return (s - lo) / (hi - lo) * 100

def already_has(row):
    """Row is done if all TARGET_COLS are filled."""
    for c in TARGET_COLS:
        v = row.get(c)
        if v is None or (isinstance(v, float) and pd.isna(v)):
            return False
    return True

# ---------------- MAIN ----------------
def main():
    df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")
    for c in TARGET_COLS:
        if c not in df.columns:
            df[c] = None

    total = len(df)
    print(f"Total rows: {total}")

    # ---- 1. biz: host listing count (from dataset itself; no scraping) ----
    print("\n[1/4] Computing 'biz' (host listing count)…")
    host_counts = df["HostName"].fillna("__unknown__").value_counts()
    df["biz"] = df["HostName"].map(host_counts).fillna(1).astype("Int64")
    df.loc[df["biz"] > 1, "biz"] = 1          # binary: 1 = commercial operator
    df.loc[df["HostName"].isna(), "biz"] = pd.NA
    print(f"  biz=1 count: {int((df['biz'] == 1).sum())}")
    print(f"  biz=0 count: {int((df['biz'] == 0).sum())}")

    # ---- 2. dist: to city centre (from dataset lat/lng) ----
    print("\n[2/4] Computing 'dist' (haversine to city centre)…")
    for i, row in df.iterrows():
        lat, lng = row.get("Latitude"), row.get("Longitude")
        if pd.isna(lat) or pd.isna(lng):
            continue
        city = str(row.get("city", "")).strip()
        centre = CITY_CENTRES.get(city)
        if centre is None:
            sub = df[df["city"] == city][["Latitude","Longitude"]].dropna()
            if len(sub):
                centre = (sub["Latitude"].median(), sub["Longitude"].median())
        if centre is None:
            continue
        df.at[i, "dist"] = round(haversine_km(lat, lng, *centre), 3)
    print(f"  filled: {int(df['dist'].notna().sum())}/{total}")

    # ---- 3. per-row: rating (URL) + Overpass features ----
    print("\n[3/4] Per-row scraping (rating from URL) + geo enrichment…")
    todo = [i for i in df.index if not already_has(df.loc[i])]
    if FORCE_REFRESH:
        todo = list(df.index)
    print(f"  rows to process: {len(todo)}")

    driver = Driver(browser="chrome", headless=HEADLESS)
    driver.set_window_size(1920, 1080)
    wait = WebDriverWait(driver, 10)

    try:
        for n, i in enumerate(todo, 1):
            url = df.at[i, "URL"]
            if not isinstance(url, str) or "airbnb" not in url:
                print(f"[{n}/{len(todo)}] row {i}: no URL, skip")
                continue

            lat = df.at[i, "Latitude"]
            lng = df.at[i, "Longitude"]

            # (a) rating from URL
            rating = None
            try:
                driver.get(url)
                wait.until(EC.presence_of_element_located((By.TAG_NAME, "h1")))
                time.sleep(random.uniform(*SLEEP_BETWEEN))
                ld = get_json_ld(driver)
                rating = rating_from_ld(ld)
            except Exception as e:
                print(f"[{n}/{len(todo)}] row {i}: rating scrape failed: {type(e).__name__}")

            # fallback: Rating from CSV
            if rating is None:
                r = df.at[i, "Rating"] if "Rating" in df.columns else None
                if pd.notna(r):
                    rating = float(r)
            if rating is not None:
                df.at[i, "guest_satisfaction_overall"] = round(rating * 20, 2)

            # (b) geo features from lat/lng
            if pd.notna(lat) and pd.notna(lng):
                try:
                    geo = get_geo_features(lat, lng)
                    df.at[i, "metro_dist"]  = geo["metro_dist"]
                    df.at[i, "attr_index"]  = geo["attr_index"]
                    df.at[i, "rest_index"]  = geo["rest_index"]
                except Exception as e:
                    print(f"[{n}/{len(todo)}] row {i}: overpass failed: {type(e).__name__}")

            print(f"[{n}/{len(todo)}] row {i:>3} | rating={df.at[i,'guest_satisfaction_overall']} "
                  f"| dist={df.at[i,'dist']} | metro={df.at[i,'metro_dist']} "
                  f"| attr={df.at[i,'attr_index']} | rest={df.at[i,'rest_index']}")

            if n % CHECKPOINT_EVERY == 0:
                save_cache(OVERPASS_CACHE_DATA)
                df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
                print(f"   ↳ checkpoint saved ({n}/{len(todo)})")
    finally:
        try:
            driver.quit()
        except Exception:
            pass
        save_cache(OVERPASS_CACHE_DATA)

    # ---- 4. normalise ----
    print("\n[4/4] Normalising attr_index_norm / rest_index_norm…")
    df["attr_index_norm"] = minmax(df["attr_index"]).round(2)
    df["rest_index_norm"] = minmax(df["rest_index"]).round(2)

    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"\nDONE → {OUTPUT_CSV}")
    for c in TARGET_COLS:
        print(f"  {c:<28} filled: {int(df[c].notna().sum())}/{total}")

if __name__ == "__main__":
    main()

Total rows: 500

[1/4] Computing 'biz' (host listing count)…
  biz=1 count: 499
  biz=0 count: 0

[2/4] Computing 'dist' (haversine to city centre)…
  filled: 499/500

[3/4] Per-row scraping (rating from URL) + geo enrichment…
  rows to process: 500


*** Getting chromedriver 155.0.8059.84 (Latest Stable) 

https://storage.googleapis.com/chrome-for-testing-public/155.0.8059.84/linux64/chromedriver-linux64.zip ...
Download Complete!

Extracting ['chromedriver'] from chromedriver-linux64.zip:
Unzip Complete!

['chromedriver'] was saved to:
/usr/local/lib/python3.13/dist-packages/seleniumbase/drivers/
chromedriver

[chromedriver 155.0.8059.84] is ready for use!

[1/500] row   0 | rating=99.4 | dist=1.596 | metro=None | attr=None | rest=None
[2/500] row   1 | rating=94.8 | dist=4.954 | metro=None | attr=None | rest=None
[3/500] row   2 | rating=100.0 | dist=19.868 | metro=None | attr=None | rest=None
[4/500] row   3 | rating=72.0 | dist=2.725 | metro=None | attr=None | rest=None
[5/500] ro

In [ ]:
# =====================================================
#  ENRICH PRICED CSV  (fixed: tiled Overpass, fast URL)
#  Columns added:
#    biz, guest_satisfaction_overall, dist, metro_dist,
#    attr_index, attr_index_norm, rest_index, rest_index_norm
# =====================================================
import os, json, time, math, random
from datetime import datetime

import numpy as np
import pandas as pd
import requests
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from seleniumbase import Driver

# ---------------- CONFIG ----------------
INPUT_CSV       = "/content/drive/MyDrive/my.flutter.2025/Instant_ai_training/SPRINT_01/airbnb_listings_weekdays_priced.csv"
OUTPUT_CSV      = "/content/drive/MyDrive/my.flutter.2025/Instant_ai_training/SPRINT_01/airbnb_listings_weekdays_priced_enriched.csv"
POI_CACHE       = "/content/poi_cache_v2.json"

USE_CSV_RATING  = False     # set True → skip URL entirely, use Rating×20 (finishes in ~2 min)
FORCE_REFRESH   = False     # True → ignore POI cache
HEADLESS        = True

SLEEP_BETWEEN   = (1.0, 2.5)
PAGE_LOAD_TIMEOUT = 20      # ⭐ hard cap per URL (was ~300 s!)
URL_WAIT_JSONLD   = 6       # seconds to wait for JSON-LD script tag
FAILSTREAK_LIMIT  = 10      # ⭐ after N consecutive URL failures, fall back to CSV

OVERPASS_ENDPOINTS = [
    "https://overpass.kumi.systems/api/interpreter",   # most reliable
    "https://overpass-api.de/api/interpreter",
    "https://overpass.private.coffee/api/interpreter",
]
OVERPASS_SLEEP   = 0.4
POI_RADIUS_KM    = 1.5
TILE_DEG         = 0.05    # ⭐ ~5.5 km tiles → keeps Overpass queries tiny
CITY_BUFFER_DEG  = 0.03

CITY_CENTRES = {
    "Amsterdam": (52.3676, 4.9041),  "Athens":    (37.9838, 23.7275),
    "Barcelona": (41.3874, 2.1686),  "Berlin":    (52.5200, 13.4050),
    "Budapest":  (47.4979, 19.0402), "Lisbon":    (38.7223, -9.1393),
    "London":    (51.5074, -0.1278), "Paris":     (48.8566, 2.3522),
    "Rome":      (41.9028, 12.4964), "Vienna":    (48.2082, 16.3738),
}

TARGET_COLS = ["biz","guest_satisfaction_overall","dist","metro_dist",
               "attr_index","attr_index_norm",
               "rest_index","rest_index_norm"]

# ---------------- UTIL ----------------
def haversine_km(lat1, lon1, lat2, lon2):
    R=6371.0
    p1,p2=math.radians(lat1),math.radians(lat2)
    dp=math.radians(lat2-lat1); dl=math.radians(lon2-lon1)
    a=math.sin(dp/2)**2+math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2*R*math.asin(math.sqrt(a))

def haversine_vec(lat1, lon1, lats, lons):
    if not len(lats): return np.array([])
    R=6371.0
    lat1r=math.radians(lat1); lon1r=math.radians(lon1)
    latr=np.radians(np.asarray(lats,float)); lonr=np.radians(np.asarray(lons,float))
    dlat=latr-lat1r; dlon=lonr-lon1r
    a=np.sin(dlat/2)**2+np.cos(lat1r)*np.cos(latr)*np.sin(dlon/2)**2
    return 2*R*np.arcsin(np.sqrt(a))

def minmax(s):
    s=pd.to_numeric(s,errors="coerce")
    lo,hi=s.min(),s.max()
    if pd.isna(lo) or hi==lo: return s*0
    return (s-lo)/(hi-lo)*100

# ---------------- OVERPASS ----------------
def _load_cache():
    if os.path.exists(POI_CACHE):
        try:
            with open(POI_CACHE) as f: return json.load(f)
        except Exception: return {}
    return {}

def _save_cache(c):
    try:
        with open(POI_CACHE,"w") as f: json.dump(c,f)
    except Exception: pass

CACHE = _load_cache()

def _overpass(query, timeout=30, retries=2):
    for url in OVERPASS_ENDPOINTS:
        for attempt in range(retries):
            try:
                r = requests.post(url, data={"data": query}, timeout=timeout)
                if r.status_code == 200:
                    return r.json()
                if r.status_code in (429, 504):
                    break           # move to next mirror
            except Exception:
                pass
            time.sleep(1.5 * (attempt + 1))
    return None

def _coords_from(data):
    out=[]
    if data:
        for el in data.get("elements", []):
            la=el.get("lat") or (el.get("center") or {}).get("lat")
            lo=el.get("lon") or (el.get("center") or {}).get("lon")
            if la is not None and lo is not None:
                out.append([la,lo])
    return out

def _tiles(lat_min, lat_max, lng_min, lng_max):
    tiles=[]; la=lat_min
    while la < lat_max:
        lo=lng_min
        while lo < lng_max:
            tiles.append((la, lo, min(la+TILE_DEG, lat_max), min(lo+TILE_DEG, lng_max)))
            lo += TILE_DEG
        la += TILE_DEG
    return tiles

def fetch_city_pois(city, sub):
    key=f"pois_v2::{city}"
    if key in CACHE and not FORCE_REFRESH:
        c = CACHE[key]
        print(f"   {city}: cached (attr={len(c['attr'])}, rest={len(c['rest'])}, metro={len(c['metro'])})")
        return c

    lats=sub["Latitude"].dropna(); lngs=sub["Longitude"].dropna()
    if len(lats)==0:
        return {"attr": [], "rest": [], "metro": []}

    tiles = _tiles(float(lats.min())-CITY_BUFFER_DEG, float(lats.max())+CITY_BUFFER_DEG,
                   float(lngs.min())-CITY_BUFFER_DEG, float(lngs.max())+CITY_BUFFER_DEG)
    print(f"   {city}: {len(tiles)} tiles")

    specs = {
        # ⭐ node-only + smaller tagset = 10–50× smaller Overpass payloads
        "attr":  '["tourism"~"attraction|museum|viewpoint|gallery"]',
        "rest":  '["amenity"~"restaurant|cafe|bar"]',
        "metro": '["railway"="station"]["station"="subway"]',
    }
    out = {k: [] for k in specs}

    for kind, tags in specs.items():
        for (la1,lo1,la2,lo2) in tiles:
            q = f'[out:json][timeout:20];node{tags}({la1},{lo1},{la2},{lo2});out body;'
            out[kind].extend(_coords_from(_overpass(q, timeout=25)))
            time.sleep(OVERPASS_SLEEP)
        # add subway entrances separately
        if kind == "metro":
            for (la1,lo1,la2,lo2) in tiles:
                q = f'[out:json][timeout:20];node["railway"="subway_entrance"]({la1},{lo1},{la2},{lo2});out body;'
                out[kind].extend(_coords_from(_overpass(q, timeout=25)))
                time.sleep(OVERPASS_SLEEP)
        print(f"   {city}: {kind} = {len(out[kind])} POIs")

    CACHE[key]=out; _save_cache(CACHE)
    return out

# ---------------- URL SCRAPER ----------------
def _json_ld(driver):
    out=[]
    try:
        for s in driver.find_elements(By.CSS_SELECTOR,'script[type="application/ld+json"]'):
            try: out.append(json.loads(s.get_attribute("textContent")))
            except Exception: pass
    except Exception: pass
    return out

def _rating_from_ld(ld):
    for obj in ld:
        if not isinstance(obj, dict): continue
        agg=obj.get("aggregateRating")
        if isinstance(agg, dict):
            rv=agg.get("ratingValue")
            if rv is not None:
                try:
                    v=float(rv)
                    if v>0: return v
                except Exception: pass
    return None

def make_driver():
    # ⭐ the three lines that fix the 6-min-per-row hang
    d = Driver(
        browser="chrome",
        headless=HEADLESS,
        page_load_strategy="eager",
        disable_images=True,
        ad_block_on=True,
    )
    d.set_window_size(1600, 1000)
    d.set_page_load_timeout(PAGE_LOAD_TIMEOUT)
    return d

def fetch_rating(driver, url):
    try:
        driver.get(url)
    except TimeoutException:
        # page didn't finish loading — but JSON-LD is usually in <head>, already there
        try: driver.execute_script("window.stop();")
        except Exception: pass
    except WebDriverException:
        return None

    # ⭐ wait for JSON-LD tag, not <h1> — the <h1> is client-rendered and slow
    try:
        WebDriverWait(driver, URL_WAIT_JSONLD).until(
            EC.presence_of_element_located((By.CSS_SELECTOR,'script[type="application/ld+json"]'))
        )
    except TimeoutException:
        pass
    return _rating_from_ld(_json_ld(driver))

# ---------------- MAIN ----------------
def main():
    df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")
    for c in TARGET_COLS:
        if c not in df.columns: df[c]=None
    total = len(df)
    print(f"Total rows: {total}\n")

    # --- 1. biz ---
    print("[1/5] biz (host has >1 listing)…")
    hc = df["HostName"].fillna("__unknown__").value_counts()
    df["biz"] = df["HostName"].map(hc).fillna(1).astype("Int64").clip(upper=1)
    df.loc[df["HostName"].isna(),"biz"]=pd.NA
    print(f"  biz=1: {int((df['biz']==1).sum())}   biz=0: {int((df['biz']==0).sum())}")

    # --- 2. dist ---
    print("\n[2/5] dist (km to city centre)…")
    for i,row in df.iterrows():
        lat,lng=row.get("Latitude"),row.get("Longitude")
        if pd.isna(lat) or pd.isna(lng): continue
        city=str(row.get("city","")).strip()
        centre=CITY_CENTRES.get(city)
        if centre is None:
            sub=df[df["city"]==city][["Latitude","Longitude"]].dropna()
            if len(sub): centre=(sub["Latitude"].median(),sub["Longitude"].median())
        if centre is None: continue
        df.at[i,"dist"]=round(haversine_km(lat,lng,*centre),3)
    print(f"  filled: {int(df['dist'].notna().sum())}/{total}")

    # --- 3. POIs per city (tiled Overpass) ---
    print("\n[3/5] Fetching POIs per city (tiled Overpass)…")
    city_pois={}
    for city in df["city"].dropna().unique():
        city_pois[city]=fetch_city_pois(city, df[df["city"]==city])

    # --- 4. Per-row: rating (URL or CSV) + geo ---
    print("\n[4/5] Per-row: rating + geo features…")
    need_url = (not USE_CSV_RATING)
    driver = make_driver() if need_url else None
    fail_streak = 0
    url_disabled = False

    try:
        for n,i in enumerate(df.index,1):
            lat=df.at[i,"Latitude"]; lng=df.at[i,"Longitude"]
            city=df.at[i,"city"]

            # (a) rating
            if (not url_disabled) and pd.isna(df.at[i,"guest_satisfaction_overall"]):
                url=df.at[i,"URL"]; rating=None
                if isinstance(url,str) and "airbnb" in url and driver:
                    rating=fetch_rating(driver,url)
                    time.sleep(random.uniform(*SLEEP_BETWEEN))

                if rating is None:
                    fail_streak+=1
                    if fail_streak>=FAILSTREAK_LIMIT and not url_disabled:
                        print(f"⚠️  {FAILSTREAK_LIMIT} URL failures in a row — "
                              f"switching to CSV Rating for the rest.")
                        url_disabled=True
                else:
                    fail_streak=0
                    df.at[i,"guest_satisfaction_overall"]=round(rating*20,2)

                if rating is None and "Rating" in df.columns and pd.notna(df.at[i,"Rating"]):
                    df.at[i,"guest_satisfaction_overall"]=round(float(df.at[i,"Rating"])*20,2)

            # (b) geo
            pois=city_pois.get(city) or {"attr":[],"rest":[],"metro":[]}
            if pd.notna(lat) and pd.notna(lng):
                if pois["metro"]:
                    d=haversine_vec(lat,lng,[p[0] for p in pois["metro"]],
                                         [p[1] for p in pois["metro"]])
                    df.at[i,"metro_dist"]=round(float(d.min()),3)
                else:
                    df.at[i,"metro_dist"]=np.nan

                if pois["attr"]:
                    d=haversine_vec(lat,lng,[p[0] for p in pois["attr"]],
                                         [p[1] for p in pois["attr"]])
                    df.at[i,"attr_index"]=int((d<=POI_RADIUS_KM).sum())
                else:
                    df.at[i,"attr_index"]=0

                if pois["rest"]:
                    d=haversine_vec(lat,lng,[p[0] for p in pois["rest"]],
                                         [p[1] for p in pois["rest"]])
                    df.at[i,"rest_index"]=int((d<=POI_RADIUS_KM).sum())
                else:
                    df.at[i,"rest_index"]=0

            if n%25==0 or n==total:
                print(f"[{n}/{total}] row {i:>3} | "
                      f"rating={df.at[i,'guest_satisfaction_overall']} | "
                      f"dist={df.at[i,'dist']} | metro={df.at[i,'metro_dist']} | "
                      f"attr={df.at[i,'attr_index']} | rest={df.at[i,'rest_index']}")

            if n%CHECKPOINT_EVERY==0 if 'CHECKPOINT_EVERY' in dir() else n%25==0:
                df.to_csv(OUTPUT_CSV,index=False,encoding="utf-8-sig")
                _save_cache(CACHE)
    finally:
        if driver:
            try: driver.quit()
            except Exception: pass
        _save_cache(CACHE)

    # --- 5. normalise ---
    print("\n[5/5] Normalising…")
    df["attr_index_norm"]=minmax(df["attr_index"]).round(2)
    df["rest_index_norm"]=minmax(df["rest_index"]).round(2)

    df.to_csv(OUTPUT_CSV,index=False,encoding="utf-8-sig")
    print(f"\nDONE → {OUTPUT_CSV}")
    for c in TARGET_COLS:
        print(f"  {c:<28} filled: {int(df[c].notna().sum())}/{total}")

if __name__=="__main__":
    main()

Total rows: 500

[1/5] biz (host has >1 listing)…
  biz=1: 499   biz=0: 0

[2/5] dist (km to city centre)…
  filled: 499/500

[3/5] Fetching POIs per city (tiled Overpass)…
   Amsterdam: 104 tiles
   Amsterdam: attr = 13 POIs
   Amsterdam: rest = 145 POIs
   Amsterdam: metro = 0 POIs
   Athens: 12 tiles
   Athens: attr = 0 POIs
   Athens: rest = 0 POIs
   Athens: metro = 0 POIs
   Barcelona: 12 tiles
   Barcelona: attr = 0 POIs
   Barcelona: rest = 0 POIs
   Barcelona: metro = 0 POIs
   Berlin: 56 tiles
   Berlin: attr = 0 POIs
   Berlin: rest = 0 POIs
   Berlin: metro = 0 POIs
   Budapest: 6 tiles
   Budapest: attr = 0 POIs
   Budapest: rest = 0 POIs
   Budapest: metro = 0 POIs
   Lisbon: 12 tiles
   Lisbon: attr = 0 POIs
   Lisbon: rest = 0 POIs
   Lisbon: metro = 3 POIs
   London: 45 tiles
   London: attr = 0 POIs
   London: rest = 0 POIs
   London: metro = 0 POIs
   Paris: 32 tiles
   Paris: attr = 0 POIs
   Paris: rest = 0 POIs
   Paris: metro = 0 POIs
   Rome: 8 tiles
   Rome: at

TypeError: Driver() got an unexpected keyword argument 'disable_images'. Did you mean 'disable_cookies'?

In [ ]:
# =====================================================
#  ENRICH PRICED CSV (v3 — Overpass rate-limit-safe)
#  Adds:
#    biz, guest_satisfaction_overall, dist, metro_dist,
#    attr_index, attr_index_norm, rest_index, rest_index_norm
# =====================================================
import os, json, time, math, random, inspect

import numpy as np
import pandas as pd
import requests
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from seleniumbase import Driver

# ---------------- CONFIG ----------------
INPUT_CSV       = "/content/drive/MyDrive/my.flutter.2025/Instant_ai_training/SPRINT_01/airbnb_listings_weekdays_priced.csv"
OUTPUT_CSV      = "/content/drive/MyDrive/my.flutter.2025/Instant_ai_training/SPRINT_01/airbnb_listings_weekdays_priced_enriched.csv"
POI_CACHE       = "/content/drive/MyDrive/my.flutter.2025/Instant_ai_training/SPRINT_01/poi_cache_v3.json"

USE_CSV_RATING  = False          # True → skip URL, use Rating×20 (finishes in ~2 min)
FORCE_REFRESH   = True           # True → ignore POI cache (needed for first clean run)
HEADLESS        = True

SLEEP_BETWEEN      = (1.0, 2.5)
PAGE_LOAD_TIMEOUT  = 20
URL_WAIT_JSONLD    = 6
FAILSTREAK_LIMIT   = 10
CHECKPOINT_EVERY   = 25

OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",      # primary (largest quota)
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.private.coffee/api/interpreter",
]
OVERPASS_SLEEP   = 1.0           # 1 s between requests (was 0.4 — too fast)
OVERPASS_RETRIES = 3             # per endpoint, exponential 2/4/8 s
POI_RADIUS_KM    = 1.5
TILE_DEG         = 0.06          # bigger tiles, fewer requests
CITY_BUFFER_DEG  = 0.02

CITY_CENTRES = {
    "Amsterdam": (52.3676, 4.9041),  "Athens":    (37.9838, 23.7275),
    "Barcelona": (41.3874, 2.1686),  "Berlin":    (52.5200, 13.4050),
    "Budapest":  (47.4979, 19.0402), "Lisbon":    (38.7223, -9.1393),
    "London":    (51.5074, -0.1278), "Paris":     (48.8566, 2.3522),
    "Rome":      (41.9028, 12.4964), "Vienna":    (48.2082, 16.3738),
}

TARGET_COLS = ["biz","guest_satisfaction_overall","dist","metro_dist",
               "attr_index","attr_index_norm",
               "rest_index","rest_index_norm"]

# ---------------- GEOMETRY ----------------
def haversine_km(lat1, lon1, lat2, lon2):
    R=6371.0
    p1,p2=math.radians(lat1),math.radians(lat2)
    dp=math.radians(lat2-lat1); dl=math.radians(lon2-lon1)
    a=math.sin(dp/2)**2+math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2*R*math.asin(math.sqrt(a))

def haversine_vec(lat1, lon1, lats, lons):
    if not len(lats): return np.array([])
    R=6371.0
    lat1r=math.radians(lat1); lon1r=math.radians(lon1)
    latr=np.radians(np.asarray(lats,float)); lonr=np.radians(np.asarray(lons,float))
    dlat=latr-lat1r; dlon=lonr-lon1r
    a=np.sin(dlat/2)**2+np.cos(lat1r)*np.cos(latr)*np.sin(dlon/2)**2
    return 2*R*np.arcsin(np.sqrt(a))

def minmax(s):
    s=pd.to_numeric(s,errors="coerce")
    lo,hi=s.min(),s.max()
    if pd.isna(lo) or hi==lo: return s*0
    return (s-lo)/(hi-lo)*100

# ---------------- OVERPASS ----------------
def _load_cache():
    if os.path.exists(POI_CACHE):
        try:
            with open(POI_CACHE) as f: return json.load(f)
        except Exception: return {}
    return {}

def _save_cache(c):
    try:
        with open(POI_CACHE,"w") as f: json.dump(c,f)
    except Exception: pass

CACHE = _load_cache()

def _overpass(query, timeout=30, retries=OVERPASS_RETRIES):
    """Rotate endpoints × exponential backoff. Returns dict or None."""
    last_err = None
    for url in OVERPASS_ENDPOINTS:
        for attempt in range(retries):
            try:
                r = requests.post(url, data={"data": query}, timeout=timeout)
                if r.status_code == 200:
                    return r.json()
                last_err = f"HTTP {r.status_code}"
                if r.status_code == 400:
                    return None   # malformed — retrying won't help
            except Exception as e:
                last_err = f"{type(e).__name__}"
            time.sleep(2 * (attempt + 1))    # 2, 4, 8 s
    return None

def _coords_from(data):
    if not data: return []
    out=[]
    for el in data.get("elements", []):
        la=el.get("lat") or (el.get("center") or {}).get("lat")
        lo=el.get("lon") or (el.get("center") or {}).get("lon")
        if la is not None and lo is not None:
            out.append([la,lo])
    return out

def _tiles_with_listings(pts, lat_min, lat_max, lng_min, lng_max):
    """Only tiles that actually contain ≥1 listing → 5–10× fewer requests."""
    tiles=[]
    la=lat_min
    while la < lat_max:
        lo=lng_min
        while lo < lng_max:
            la2=min(la+TILE_DEG, lat_max)
            lo2=min(lo+TILE_DEG, lng_max)
            if any(la <= p[0] < la2 and lo <= p[1] < lo2 for p in pts):
                tiles.append((la,lo,la2,lo2))
            lo += TILE_DEG
        la += TILE_DEG
    return tiles

def fetch_city_pois(city, sub):
    key=f"pois_v3::{city}"
    if not FORCE_REFRESH and key in CACHE:
        c=CACHE[key]
        print(f"   {city}: cached (attr={len(c['attr'])}, "
              f"rest={len(c['rest'])}, metro={len(c['metro'])})")
        return c

    pts = sub[["Latitude","Longitude"]].dropna().values.tolist()
    if not pts:
        return {"attr":[], "rest":[], "metro":[]}

    lats=[p[0] for p in pts]; lngs=[p[1] for p in pts]
    tiles = _tiles_with_listings(
        pts,
        min(lats)-CITY_BUFFER_DEG, max(lats)+CITY_BUFFER_DEG,
        min(lngs)-CITY_BUFFER_DEG, max(lngs)+CITY_BUFFER_DEG,
    )
    print(f"   {city}: {len(tiles)} tiles (filtered to listings)")

    specs = {
        # nwr = nodes + ways + relations → catches ways-mapped POIs too
        "attr":  '["tourism"~"attraction|museum|viewpoint|gallery"]',
        "rest":  '["amenity"~"restaurant|cafe|bar"]',
        "metro": '["railway"="station"]["station"="subway"]',
    }
    out = {k: [] for k in specs}
    errors = {k: 0 for k in specs}

    for kind, tags in specs.items():
        for (la1,lo1,la2,lo2) in tiles:
            q = f'[out:json][timeout:25];nwr{tags}({la1},{lo1},{la2},{lo2});out center;'
            data = _overpass(q, timeout=30)
            if data is None:
                errors[kind] += 1
            else:
                out[kind].extend(_coords_from(data))
            time.sleep(OVERPASS_SLEEP)
        # subway entrances separately (different tag)
        if kind == "metro":
            for (la1,lo1,la2,lo2) in tiles:
                q = (f'[out:json][timeout:20];'
                     f'nwr["railway"="subway_entrance"]({la1},{lo1},{la2},{lo2});out center;')
                data = _overpass(q, timeout=25)
                if data is None:
                    errors[kind] += 1
                else:
                    out[kind].extend(_coords_from(data))
                time.sleep(OVERPASS_SLEEP)

        fail_pct = errors[kind] / max(len(tiles), 1) * 100
        note = f"  ⚠️ {fail_pct:.0f}% tiles failed" if fail_pct > 20 else ""
        print(f"   {city}: {kind} = {len(out[kind])} POIs{note}")

    # only cache if we got SOME data — otherwise retry next run
    if any(len(v) for v in out.values()):
        CACHE[key] = out
        _save_cache(CACHE)
    else:
        print(f"   ⚠️ {city}: Overpass returned 0 everywhere — not cached, rerun to retry")
    return out

# ---------------- URL SCRAPER ----------------
def _json_ld(driver):
    out=[]
    try:
        for s in driver.find_elements(By.CSS_SELECTOR,'script[type="application/ld+json"]'):
            try: out.append(json.loads(s.get_attribute("textContent")))
            except Exception: pass
    except Exception: pass
    return out

def _rating_from_ld(ld):
    for obj in ld:
        if not isinstance(obj, dict): continue
        agg=obj.get("aggregateRating")
        if isinstance(agg, dict):
            rv=agg.get("ratingValue")
            if rv is not None:
                try:
                    v=float(rv)
                    if v >= 1.0:       # 0.0 means "no reviews" — ignore
                        return v
                except Exception: pass
    return None

def make_driver():
    """Pass only kwargs this seleniumbase version accepts."""
    sig = inspect.signature(Driver.__init__).parameters
    candidates = {
        "browser":            "chrome",
        "headless":           HEADLESS,
        "page_load_strategy": "eager",   # ⭐ 4× faster than normal
        "block_images":       True,      # ⭐ correct kwarg (was disable_images)
        "ad_block_on":        True,
    }
    kwargs = {k:v for k,v in candidates.items() if k in sig}
    print(f"[driver] using kwargs: {list(kwargs)}")
    d = Driver(**kwargs)
    d.set_window_size(1600, 1000)
    d.set_page_load_timeout(PAGE_LOAD_TIMEOUT)
    return d

def fetch_rating(driver, url):
    try:
        driver.get(url)
    except TimeoutException:
        try: driver.execute_script("window.stop();")
        except Exception: pass
    except WebDriverException:
        return None
    try:
        WebDriverWait(driver, URL_WAIT_JSONLD).until(
            EC.presence_of_element_located((By.CSS_SELECTOR,'script[type="application/ld+json"]'))
        )
    except TimeoutException:
        pass
    return _rating_from_ld(_json_ld(driver))

# ---------------- MAIN ----------------
def main():
    df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")
    for c in TARGET_COLS:
        if c not in df.columns: df[c]=None
    total=len(df)
    print(f"Total rows: {total}\n")

    # --- 1. biz ---
    print("[1/5] biz (host has >1 listing)…")
    hc = df["HostName"].fillna("__unknown__").value_counts()
    df["biz"] = df["HostName"].map(hc).fillna(1).astype("Int64").clip(upper=1)
    df.loc[df["HostName"].isna(),"biz"]=pd.NA
    print(f"  biz=1: {int((df['biz']==1).sum())}   biz=0: {int((df['biz']==0).sum())}")

    # --- 2. dist ---
    print("\n[2/5] dist (km to city centre)…")
    for i,row in df.iterrows():
        lat,lng=row.get("Latitude"),row.get("Longitude")
        if pd.isna(lat) or pd.isna(lng): continue
        city=str(row.get("city","")).strip()
        centre=CITY_CENTRES.get(city)
        if centre is None:
            sub=df[df["city"]==city][["Latitude","Longitude"]].dropna()
            if len(sub): centre=(sub["Latitude"].median(),sub["Longitude"].median())
        if centre is None: continue
        df.at[i,"dist"]=round(haversine_km(lat,lng,*centre),3)
    print(f"  filled: {int(df['dist'].notna().sum())}/{total}")

    # --- 3. POIs per city ---
    print("\n[3/5] Fetching POIs per city…")
    city_pois={}
    for city in df["city"].dropna().unique():
        city_pois[city]=fetch_city_pois(city, df[df["city"]==city])

    # --- 4. per-row: rating + geo ---
    print("\n[4/5] Per-row: rating + geo features…")
    need_url = not USE_CSV_RATING
    driver = make_driver() if need_url else None
    fail_streak = 0
    url_disabled = False

    try:
        for n,i in enumerate(df.index,1):
            lat=df.at[i,"Latitude"]; lng=df.at[i,"Longitude"]
            city=df.at[i,"city"]

            # (a) rating
            if (not url_disabled) and pd.isna(df.at[i,"guest_satisfaction_overall"]):
                url=df.at[i,"URL"]; rating=None
                if isinstance(url,str) and "airbnb" in url and driver:
                    rating=fetch_rating(driver,url)
                    time.sleep(random.uniform(*SLEEP_BETWEEN))

                if rating is None:
                    fail_streak+=1
                    if fail_streak>=FAILSTREAK_LIMIT and not url_disabled:
                        print(f"⚠️  {FAILSTREAK_LIMIT} URL failures in a row — "
                              f"switching to CSV Rating×20 for remaining rows.")
                        url_disabled=True
                else:
                    fail_streak=0
                    df.at[i,"guest_satisfaction_overall"]=round(rating*20,2)

                # fallback to CSV Rating
                if rating is None and "Rating" in df.columns and pd.notna(df.at[i,"Rating"]):
                    df.at[i,"guest_satisfaction_overall"]=round(float(df.at[i,"Rating"])*20,2)

            # (b) geo
            pois=city_pois.get(city) or {"attr":[],"rest":[],"metro":[]}
            if pd.notna(lat) and pd.notna(lng):
                if pois["metro"]:
                    d=haversine_vec(lat,lng,[p[0] for p in pois["metro"]],
                                         [p[1] for p in pois["metro"]])
                    df.at[i,"metro_dist"]=round(float(d.min()),3)
                else:
                    df.at[i,"metro_dist"]=np.nan

                if pois["attr"]:
                    d=haversine_vec(lat,lng,[p[0] for p in pois["attr"]],
                                         [p[1] for p in pois["attr"]])
                    df.at[i,"attr_index"]=int((d<=POI_RADIUS_KM).sum())
                else:
                    df.at[i,"attr_index"]=0

                if pois["rest"]:
                    d=haversine_vec(lat,lng,[p[0] for p in pois["rest"]],
                                         [p[1] for p in pois["rest"]])
                    df.at[i,"rest_index"]=int((d<=POI_RADIUS_KM).sum())
                else:
                    df.at[i,"rest_index"]=0

            if n%25==0 or n==total:
                print(f"[{n}/{total}] row {i:>3} | "
                      f"rating={df.at[i,'guest_satisfaction_overall']} | "
                      f"dist={df.at[i,'dist']} | metro={df.at[i,'metro_dist']} | "
                      f"attr={df.at[i,'attr_index']} | rest={df.at[i,'rest_index']}")

            if n%CHECKPOINT_EVERY==0:
                df.to_csv(OUTPUT_CSV,index=False,encoding="utf-8-sig")
                _save_cache(CACHE)
    finally:
        if driver:
            try: driver.quit()
            except Exception: pass
        _save_cache(CACHE)

    # --- 5. normalise ---
    print("\n[5/5] Normalising…")
    df["attr_index_norm"]=minmax(df["attr_index"]).round(2)
    df["rest_index_norm"]=minmax(df["rest_index"]).round(2)
    df.to_csv(OUTPUT_CSV,index=False,encoding="utf-8-sig")

    print(f"\nDONE → {OUTPUT_CSV}")
    for c in TARGET_COLS:
        print(f"  {c:<28} filled: {int(df[c].notna().sum())}/{total}")


if __name__=="__main__":
    main()

Total rows: 500

[1/5] biz (host has >1 listing)…
  biz=1: 499   biz=0: 0

[2/5] dist (km to city centre)…
  filled: 499/500

[3/5] Fetching POIs per city…
   Amsterdam: 23 tiles (filtered to listings)
   Amsterdam: attr = 0 POIs  ⚠️ 100% tiles failed
   Amsterdam: rest = 0 POIs  ⚠️ 100% tiles failed
   Amsterdam: metro = 0 POIs  ⚠️ 200% tiles failed
   ⚠️ Amsterdam: Overpass returned 0 everywhere — not cached, rerun to retry
   Athens: 4 tiles (filtered to listings)
   Athens: attr = 0 POIs  ⚠️ 100% tiles failed
   Athens: rest = 0 POIs  ⚠️ 100% tiles failed
   Athens: metro = 0 POIs  ⚠️ 200% tiles failed
   ⚠️ Athens: Overpass returned 0 everywhere — not cached, rerun to retry
   Barcelona: 5 tiles (filtered to listings)
   Barcelona: attr = 0 POIs  ⚠️ 100% tiles failed
   Barcelona: rest = 0 POIs  ⚠️ 100% tiles failed
   Barcelona: metro = 0 POIs  ⚠️ 200% tiles failed
   ⚠️ Barcelona: Overpass returned 0 everywhere — not cached, rerun to retry
   Berlin: 10 tiles (filtered to listin

In [3]:
# =====================================================
#  ENRICH PRICED CSV (v3 — Overpass rate-limit-safe)
#  Adds:
#    biz, guest_satisfaction_overall, dist, metro_dist,
#    attr_index, attr_index_norm, rest_index, rest_index_norm
# =====================================================
import os, json, time, math, random, inspect

import numpy as np
import pandas as pd
import requests
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from seleniumbase import Driver

# ---------------- CONFIG ----------------
INPUT_CSV       = "/content/drive/MyDrive/my.flutter.2025/Instant_ai_training/SPRINT_01/airbnb_listings_weekends_priced.csv"
OUTPUT_CSV      = "/content/drive/MyDrive/my.flutter.2025/Instant_ai_training/SPRINT_01/airbnb_listings_weekends_priced_enriched.csv"
POI_CACHE       = "/content/drive/MyDrive/my.flutter.2025/Instant_ai_training/SPRINT_01/poi_cache_v3.json"

USE_CSV_RATING  = False          # True → skip URL, use Rating×20 (finishes in ~2 min)
FORCE_REFRESH   = True           # True → ignore POI cache (needed for first clean run)
HEADLESS        = True

SLEEP_BETWEEN      = (1.0, 2.5)
PAGE_LOAD_TIMEOUT  = 20
URL_WAIT_JSONLD    = 6
FAILSTREAK_LIMIT   = 10
CHECKPOINT_EVERY   = 25

OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",      # primary (largest quota)
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.private.coffee/api/interpreter",
]
OVERPASS_SLEEP   = 1.0           # 1 s between requests (was 0.4 — too fast)
OVERPASS_RETRIES = 3             # per endpoint, exponential 2/4/8 s
POI_RADIUS_KM    = 1.5
TILE_DEG         = 0.06          # bigger tiles, fewer requests
CITY_BUFFER_DEG  = 0.02

CITY_CENTRES = {
    "Amsterdam": (52.3676, 4.9041),  "Athens":    (37.9838, 23.7275),
    "Barcelona": (41.3874, 2.1686),  "Berlin":    (52.5200, 13.4050),
    "Budapest":  (47.4979, 19.0402), "Lisbon":    (38.7223, -9.1393),
    "London":    (51.5074, -0.1278), "Paris":     (48.8566, 2.3522),
    "Rome":      (41.9028, 12.4964), "Vienna":    (48.2082, 16.3738),
}

TARGET_COLS = ["biz","guest_satisfaction_overall","dist","metro_dist",
               "attr_index","attr_index_norm",
               "rest_index","rest_index_norm"]

# ---------------- GEOMETRY ----------------
def haversine_km(lat1, lon1, lat2, lon2):
    R=6371.0
    p1,p2=math.radians(lat1),math.radians(lat2)
    dp=math.radians(lat2-lat1); dl=math.radians(lon2-lon1)
    a=math.sin(dp/2)**2+math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2*R*math.asin(math.sqrt(a))

def haversine_vec(lat1, lon1, lats, lons):
    if not len(lats): return np.array([])
    R=6371.0
    lat1r=math.radians(lat1); lon1r=math.radians(lon1)
    latr=np.radians(np.asarray(lats,float)); lonr=np.radians(np.asarray(lons,float))
    dlat=latr-lat1r; dlon=lonr-lon1r
    a=np.sin(dlat/2)**2+np.cos(lat1r)*np.cos(latr)*np.sin(dlon/2)**2
    return 2*R*np.arcsin(np.sqrt(a))

def minmax(s):
    s=pd.to_numeric(s,errors="coerce")
    lo,hi=s.min(),s.max()
    if pd.isna(lo) or hi==lo: return s*0
    return (s-lo)/(hi-lo)*100

# ---------------- OVERPASS ----------------
def _load_cache():
    if os.path.exists(POI_CACHE):
        try:
            with open(POI_CACHE) as f: return json.load(f)
        except Exception: return {}
    return {}

def _save_cache(c):
    try:
        with open(POI_CACHE,"w") as f: json.dump(c,f)
    except Exception: pass

CACHE = _load_cache()

def _overpass(query, timeout=30, retries=OVERPASS_RETRIES):
    """Rotate endpoints × exponential backoff. Returns dict or None."""
    last_err = None
    for url in OVERPASS_ENDPOINTS:
        for attempt in range(retries):
            try:
                r = requests.post(url, data={"data": query}, timeout=timeout)
                if r.status_code == 200:
                    return r.json()
                last_err = f"HTTP {r.status_code}"
                if r.status_code == 400:
                    return None   # malformed — retrying won't help
            except Exception as e:
                last_err = f"{type(e).__name__}"
            time.sleep(2 * (attempt + 1))    # 2, 4, 8 s
    return None

def _coords_from(data):
    if not data: return []
    out=[]
    for el in data.get("elements", []):
        la=el.get("lat") or (el.get("center") or {}).get("lat")
        lo=el.get("lon") or (el.get("center") or {}).get("lon")
        if la is not None and lo is not None:
            out.append([la,lo])
    return out

def _tiles_with_listings(pts, lat_min, lat_max, lng_min, lng_max):
    """Only tiles that actually contain ≥1 listing → 5–10× fewer requests."""
    tiles=[]
    la=lat_min
    while la < lat_max:
        lo=lng_min
        while lo < lng_max:
            la2=min(la+TILE_DEG, lat_max)
            lo2=min(lo+TILE_DEG, lng_max)
            if any(la <= p[0] < la2 and lo <= p[1] < lo2 for p in pts):
                tiles.append((la,lo,la2,lo2))
            lo += TILE_DEG
        la += TILE_DEG
    return tiles

def fetch_city_pois(city, sub):
    key=f"pois_v3::{city}"
    if not FORCE_REFRESH and key in CACHE:
        c=CACHE[key]
        print(f"   {city}: cached (attr={len(c['attr'])}, "
              f"rest={len(c['rest'])}, metro={len(c['metro'])})")
        return c

    pts = sub[["Latitude","Longitude"]].dropna().values.tolist()
    if not pts:
        return {"attr":[], "rest":[], "metro":[]}

    lats=[p[0] for p in pts]; lngs=[p[1] for p in pts]
    tiles = _tiles_with_listings(
        pts,
        min(lats)-CITY_BUFFER_DEG, max(lats)+CITY_BUFFER_DEG,
        min(lngs)-CITY_BUFFER_DEG, max(lngs)+CITY_BUFFER_DEG,
    )
    print(f"   {city}: {len(tiles)} tiles (filtered to listings)")

    specs = {
        # nwr = nodes + ways + relations → catches ways-mapped POIs too
        "attr":  '["tourism"~"attraction|museum|viewpoint|gallery"]',
        "rest":  '["amenity"~"restaurant|cafe|bar"]',
        "metro": '["railway"="station"]["station"="subway"]',
    }
    out = {k: [] for k in specs}
    errors = {k: 0 for k in specs}

    for kind, tags in specs.items():
        for (la1,lo1,la2,lo2) in tiles:
            q = f'[out:json][timeout:25];nwr{tags}({la1},{lo1},{la2},{lo2});out center;'
            data = _overpass(q, timeout=30)
            if data is None:
                errors[kind] += 1
            else:
                out[kind].extend(_coords_from(data))
            time.sleep(OVERPASS_SLEEP)
        # subway entrances separately (different tag)
        if kind == "metro":
            for (la1,lo1,la2,lo2) in tiles:
                q = (f'[out:json][timeout:20];'
                     f'nwr["railway"="subway_entrance"]({la1},{lo1},{la2},{lo2});out center;')
                data = _overpass(q, timeout=25)
                if data is None:
                    errors[kind] += 1
                else:
                    out[kind].extend(_coords_from(data))
                time.sleep(OVERPASS_SLEEP)

        fail_pct = errors[kind] / max(len(tiles), 1) * 100
        note = f"  ⚠️ {fail_pct:.0f}% tiles failed" if fail_pct > 20 else ""
        print(f"   {city}: {kind} = {len(out[kind])} POIs{note}")

    # only cache if we got SOME data — otherwise retry next run
    if any(len(v) for v in out.values()):
        CACHE[key] = out
        _save_cache(CACHE)
    else:
        print(f"   ⚠️ {city}: Overpass returned 0 everywhere — not cached, rerun to retry")
    return out

# ---------------- URL SCRAPER ----------------
def _json_ld(driver):
    out=[]
    try:
        for s in driver.find_elements(By.CSS_SELECTOR,'script[type="application/ld+json"]'):
            try: out.append(json.loads(s.get_attribute("textContent")))
            except Exception: pass
    except Exception: pass
    return out

def _rating_from_ld(ld):
    for obj in ld:
        if not isinstance(obj, dict): continue
        agg=obj.get("aggregateRating")
        if isinstance(agg, dict):
            rv=agg.get("ratingValue")
            if rv is not None:
                try:
                    v=float(rv)
                    if v >= 1.0:       # 0.0 means "no reviews" — ignore
                        return v
                except Exception: pass
    return None

def make_driver():
    """Pass only kwargs this seleniumbase version accepts."""
    sig = inspect.signature(Driver.__init__).parameters
    candidates = {
        "browser":            "chrome",
        "headless":           HEADLESS,
        "page_load_strategy": "eager",   # ⭐ 4× faster than normal
        "block_images":       True,      # ⭐ correct kwarg (was disable_images)
        "ad_block_on":        True,
    }
    kwargs = {k:v for k,v in candidates.items() if k in sig}
    print(f"[driver] using kwargs: {list(kwargs)}")
    d = Driver(**kwargs)
    d.set_window_size(1600, 1000)
    d.set_page_load_timeout(PAGE_LOAD_TIMEOUT)
    return d

def fetch_rating(driver, url):
    try:
        driver.get(url)
    except TimeoutException:
        try: driver.execute_script("window.stop();")
        except Exception: pass
    except WebDriverException:
        return None
    try:
        WebDriverWait(driver, URL_WAIT_JSONLD).until(
            EC.presence_of_element_located((By.CSS_SELECTOR,'script[type="application/ld+json"]'))
        )
    except TimeoutException:
        pass
    return _rating_from_ld(_json_ld(driver))

# ---------------- MAIN ----------------
def main():
    df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")
    for c in TARGET_COLS:
        if c not in df.columns: df[c]=None
    total=len(df)
    print(f"Total rows: {total}\n")

    # --- 1. biz ---
    print("[1/5] biz (host has >1 listing)…")
    hc = df["HostName"].fillna("__unknown__").value_counts()
    df["biz"] = df["HostName"].map(hc).fillna(1).astype("Int64").clip(upper=1)
    df.loc[df["HostName"].isna(),"biz"]=pd.NA
    print(f"  biz=1: {int((df['biz']==1).sum())}   biz=0: {int((df['biz']==0).sum())}")

    # --- 2. dist ---
    print("\n[2/5] dist (km to city centre)…")
    for i,row in df.iterrows():
        lat,lng=row.get("Latitude"),row.get("Longitude")
        if pd.isna(lat) or pd.isna(lng): continue
        city=str(row.get("city","")).strip()
        centre=CITY_CENTRES.get(city)
        if centre is None:
            sub=df[df["city"]==city][["Latitude","Longitude"]].dropna()
            if len(sub): centre=(sub["Latitude"].median(),sub["Longitude"].median())
        if centre is None: continue
        df.at[i,"dist"]=round(haversine_km(lat,lng,*centre),3)
    print(f"  filled: {int(df['dist'].notna().sum())}/{total}")

    # --- 3. POIs per city ---
    print("\n[3/5] Fetching POIs per city…")
    city_pois={}
    for city in df["city"].dropna().unique():
        city_pois[city]=fetch_city_pois(city, df[df["city"]==city])

    # --- 4. per-row: rating + geo ---
    print("\n[4/5] Per-row: rating + geo features…")
    need_url = not USE_CSV_RATING
    driver = make_driver() if need_url else None
    fail_streak = 0
    url_disabled = False

    try:
        for n,i in enumerate(df.index,1):
            lat=df.at[i,"Latitude"]; lng=df.at[i,"Longitude"]
            city=df.at[i,"city"]

            # (a) rating
            if (not url_disabled) and pd.isna(df.at[i,"guest_satisfaction_overall"]):
                url=df.at[i,"URL"]; rating=None
                if isinstance(url,str) and "airbnb" in url and driver:
                    rating=fetch_rating(driver,url)
                    time.sleep(random.uniform(*SLEEP_BETWEEN))

                if rating is None:
                    fail_streak+=1
                    if fail_streak>=FAILSTREAK_LIMIT and not url_disabled:
                        print(f"⚠️  {FAILSTREAK_LIMIT} URL failures in a row — "
                              f"switching to CSV Rating×20 for remaining rows.")
                        url_disabled=True
                else:
                    fail_streak=0
                    df.at[i,"guest_satisfaction_overall"]=round(rating*20,2)

                # fallback to CSV Rating
                if rating is None and "Rating" in df.columns and pd.notna(df.at[i,"Rating"]):
                    df.at[i,"guest_satisfaction_overall"]=round(float(df.at[i,"Rating"])*20,2)

            # (b) geo
            pois=city_pois.get(city) or {"attr":[],"rest":[],"metro":[]}
            if pd.notna(lat) and pd.notna(lng):
                if pois["metro"]:
                    d=haversine_vec(lat,lng,[p[0] for p in pois["metro"]],
                                         [p[1] for p in pois["metro"]])
                    df.at[i,"metro_dist"]=round(float(d.min()),3)
                else:
                    df.at[i,"metro_dist"]=np.nan

                if pois["attr"]:
                    d=haversine_vec(lat,lng,[p[0] for p in pois["attr"]],
                                         [p[1] for p in pois["attr"]])
                    df.at[i,"attr_index"]=int((d<=POI_RADIUS_KM).sum())
                else:
                    df.at[i,"attr_index"]=0

                if pois["rest"]:
                    d=haversine_vec(lat,lng,[p[0] for p in pois["rest"]],
                                         [p[1] for p in pois["rest"]])
                    df.at[i,"rest_index"]=int((d<=POI_RADIUS_KM).sum())
                else:
                    df.at[i,"rest_index"]=0

            if n%25==0 or n==total:
                print(f"[{n}/{total}] row {i:>3} | "
                      f"rating={df.at[i,'guest_satisfaction_overall']} | "
                      f"dist={df.at[i,'dist']} | metro={df.at[i,'metro_dist']} | "
                      f"attr={df.at[i,'attr_index']} | rest={df.at[i,'rest_index']}")

            if n%CHECKPOINT_EVERY==0:
                df.to_csv(OUTPUT_CSV,index=False,encoding="utf-8-sig")
                _save_cache(CACHE)
    finally:
        if driver:
            try: driver.quit()
            except Exception: pass
        _save_cache(CACHE)

    # --- 5. normalise ---
    print("\n[5/5] Normalising…")
    df["attr_index_norm"]=minmax(df["attr_index"]).round(2)
    df["rest_index_norm"]=minmax(df["rest_index"]).round(2)
    df.to_csv(OUTPUT_CSV,index=False,encoding="utf-8-sig")

    print(f"\nDONE → {OUTPUT_CSV}")
    for c in TARGET_COLS:
        print(f"  {c:<28} filled: {int(df[c].notna().sum())}/{total}")


if __name__=="__main__":
    main()

Total rows: 500

[1/5] biz (host has >1 listing)…
  biz=1: 499   biz=0: 0

[2/5] dist (km to city centre)…
  filled: 499/500

[3/5] Fetching POIs per city…
   Amsterdam: 12 tiles (filtered to listings)
   Amsterdam: attr = 0 POIs  ⚠️ 100% tiles failed
   Amsterdam: rest = 0 POIs  ⚠️ 100% tiles failed
   Amsterdam: metro = 0 POIs  ⚠️ 200% tiles failed
   ⚠️ Amsterdam: Overpass returned 0 everywhere — not cached, rerun to retry
   Athens: 1 tiles (filtered to listings)
   Athens: attr = 0 POIs  ⚠️ 100% tiles failed
   Athens: rest = 0 POIs  ⚠️ 100% tiles failed
   Athens: metro = 0 POIs  ⚠️ 200% tiles failed
   ⚠️ Athens: Overpass returned 0 everywhere — not cached, rerun to retry
   Barcelona: 3 tiles (filtered to listings)
   Barcelona: attr = 0 POIs  ⚠️ 100% tiles failed
   Barcelona: rest = 0 POIs  ⚠️ 100% tiles failed
   Barcelona: metro = 0 POIs  ⚠️ 200% tiles failed
   ⚠️ Barcelona: Overpass returned 0 everywhere — not cached, rerun to retry
   Berlin: 10 tiles (filtered to listin